# Indoor 3DGS Splat Pipeline
**StrayScanner → 3DGS 학습 → Gaussian Splat `.ply` export**

## 실행 순서
```
SETUP  → 런타임 재시작
Cell 0  Drive 마운트 + GPU 확인
Cell 1  기본 패키지 설치
Cell 2b CSV / 입력 확인
Cell 2  StrayScanner 전처리
Cell 2c transforms.json 검증
Cell 2e LiDAR depth 기반 sparse point cloud 생성
Cell 3  3DGS 학습
Cell 3b Gaussian Splat `.ply` export
Cell 4  `.ply` 확인 및 최종 splat 경로 기록
```

## Drive 저장 구조
```
MyDrive/3dgs_project/
├── input/           ← StrayScanner 원본
├── processed/       ← 전처리 결과
│   ├── images/
│   ├── depth/
│   ├── normals_from_pretrain/
│   ├── depth_confidence_masks/
│   ├── transforms.json
│   ├── sparse_pointcloud.ply
│   └── logs/
└── output/          ← 학습 결과 및 최종 splat `.ply`
```

※ Mesh 추출, TSDF/IsoOctree/Poisson 변환, mesh 후처리, floor plan/navmap 생성 코드는 제거됨.


In [ ]:
# ============================================================
# SETUP: 패키지 설치 + 패치
# 완료 후 [런타임 > 런타임 다시 시작]
# ============================================================
import subprocess, sys, os, site

os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']

def run(cmd, check=True):
    return subprocess.run(cmd, shell=True, check=check)

def pip_ver(pkg):
    r = subprocess.run([sys.executable, '-m', 'pip', 'show', pkg],
                       capture_output=True, text=True)
    for l in r.stdout.splitlines():
        if l.startswith('Version:'): return l.split(': ',1)[1].strip()
    return None

def safe_read(p):
    with open(p, encoding='utf-8') as f: return f.read()
def safe_write(p, c):
    with open(p, 'w', encoding='utf-8') as f: f.write(c)
def safe_readlines(p):
    with open(p, encoding='utf-8') as f: return f.readlines()
def safe_writelines(p, ls):
    with open(p, 'w', encoding='utf-8') as f: f.writelines(ls)

print('numpy:', pip_ver('numpy'))

# 1) DN-Splatter 클론
repo = '/content/dn-splatter'
if not os.path.exists(repo):
    run(f'git clone https://github.com/maturk/dn-splatter.git {repo}')
    print('DN-Splatter 클론 OK')
else:
    print('DN-Splatter 이미 존재')

# 2) pyproject.toml 패치
toml = f'{repo}/pyproject.toml'
c = safe_read(toml)
for pkg in ['"vdbfusion",\n', '"omnidata-tools",\n', '"pytorch-lightning",\n',
            '"geffnet",\n', '"rerun-sdk",\n', '"pyrender",\n', '"PyMCubes==0.1.2",\n']:
    c = c.replace(pkg, '')
c = c.replace('"nerfstudio == 1.1.3"', '"nerfstudio >= 1.1.3"')
safe_write(toml, c)
print('pyproject.toml 패치 완료')

# 3) normals_from_pretrain.py 패치 (omnidata import guard)
nfp = f'{repo}/dn_splatter/scripts/normals_from_pretrain.py'
subprocess.run(f'cd {repo} && git checkout dn_splatter/scripts/normals_from_pretrain.py',
               shell=True, capture_output=True)
target = 'from omnidata_tools.torch.modules.midas.dpt_depth import DPTDepthModel'
lines  = safe_readlines(nfp)
if not any('DPTDepthModel = None' in l for l in lines):
    new = []
    for l in lines:
        if target in l:
            ind = ' ' * (len(l) - len(l.lstrip()))
            new += [f'{ind}try:\n', f'{ind}    {target}\n',
                    f'{ind}except ImportError:\n', f'{ind}    DPTDepthModel = None\n']
        else:
            new.append(l)
    safe_writelines(nfp, new)
    print('normals_from_pretrain.py 패치 완료')

# 4) normal_nerfstudio.py 패치
fpath = f'{repo}/dn_splatter/data/normal_nerfstudio.py'
code  = safe_read(fpath)
for old, new in [
    ('points=metadata["points3D_xyz"],',
     'points=metadata.get("points3D_xyz", torch.zeros((1,3))),'),
    ('colors=metadata["points3D_rgb"],',
     'colors=metadata.get("points3D_rgb", torch.zeros((1,3), dtype=torch.uint8)),'),
    ('if self.config.load_pcd_normals:',
     'if self.config.load_pcd_normals and "points3D_xyz" in metadata and "points3D_rgb" in metadata:'),
    ('        normal_filenames = self.get_normal_filepaths()',
     '        normal_filenames = self.get_normal_filepaths() if self.config.load_normals else []'),
]:
    code = code.replace(old, new)
safe_write(fpath, code)
print('normal_nerfstudio.py 패치 완료')

# 5) DN-Splatter 설치
run('pip uninstall dn-splatter -y', check=False)
run(f'{sys.executable} -m pip install -q -e {repo}')
run(f'{sys.executable} -m pip install -q geffnet timm')
print('DN-Splatter 설치 완료')

# 6) DSINE 설치 (AGS-Mesh normal 생성용)
dsine = '/content/DSINE'
if not os.path.exists(dsine):
    run(f'git clone https://github.com/baegwangbin/DSINE.git {dsine}')
    run(f'{sys.executable} -m pip install -q -e {dsine}')
    print('DSINE 설치 완료')
else:
    print('DSINE 이미 존재')

# 7) eval_utils.py 패치
for p in site.getsitepackages():
    cand = f'{p}/nerfstudio/utils/eval_utils.py'
    if os.path.exists(cand):
        code = safe_read(cand)
        old = 'loaded_state = torch.load(load_path, map_location="cpu")'
        new = 'loaded_state = torch.load(load_path, map_location="cpu", weights_only=False)'
        if old in code:
            safe_write(cand, code.replace(old, new))
            print('eval_utils.py 패치 완료')
        break

# 8) dn_model.py - confidence locals() 패치 (라인 직접 교체)
dm_path = f'{repo}/dn_splatter/dn_model.py'
lines = safe_read(dm_path).split('\n')

target_orig  = 'confidence_map=confidence,'
target_patched = 'confidence_map=confidence if "confidence" in locals() else None,'

patched = False
for i, l in enumerate(lines):
    if target_orig in l and 'locals' not in l:
        lines[i] = l.replace(target_orig, target_patched)
        patched = True
        break
    elif target_patched in l:
        patched = True  # 이미 패치됨
        break

safe_write(dm_path, '\n'.join(lines))
if patched:
    print('dn_model.py 패치 완료')
else:
    print('dn_model.py 패치 대상 없음 - 확인 필요')

# 9) regularization_strategy.py - confidence_map None 체크 패치
reg_path = f'{repo}/dn_splatter/regularization_strategy.py'
lines = safe_read(reg_path).split('\n')

new_lines = []
i = 0
while i < len(lines):
    l = lines[i]
    if l.strip() == 'gt_depth = torch.where(confidence_map > 0, gt_depth, 0).cuda()':
        indent = ' ' * (len(l) - len(l.lstrip()))
        new_lines.append(f'{indent}if confidence_map is not None:')
        new_lines.append(f'{indent}    gt_depth = torch.where(confidence_map > 0, gt_depth, 0).cuda()')
        new_lines.append(f'{indent}else:')
        new_lines.append(f'{indent}    gt_depth = gt_depth.cuda()')
        i += 1
    elif l.strip() == 'if confidence_map is not None:' and \
         i+1 < len(lines) and 'torch.where' in lines[i+1]:
        new_lines += lines[i:i+4]
        i += 4
    else:
        new_lines.append(l)
        i += 1

safe_write(reg_path, '\n'.join(new_lines))
if 'confidence_map is not None' in safe_read(reg_path):
    print('regularization_strategy.py 패치 완료')
else:
    print('regularization_strategy.py 패치 실패')

# 10) eval_utils.py weights_only 패치 (checkpoint 로드 호환)
for p in site.getsitepackages():
    cand = f'{p}/nerfstudio/utils/eval_utils.py'
    if os.path.exists(cand):
        code = safe_read(cand)
        for old, new in [
            ('torch.load(load_path, map_location="cpu")',
             'torch.load(load_path, map_location="cpu", weights_only=False)'),
            ('torch.load(load_path, map_location=map_location)',
             'torch.load(load_path, map_location=map_location, weights_only=False)'),
        ]:
            code = code.replace(old, new)
        safe_write(cand, code)
        print('eval_utils.py 패치 완료')
        break

# 11) trainer.py _load_checkpoint 패치
import site
for p in site.getsitepackages():
    cand = f'{p}/nerfstudio/engine/trainer.py'
    if os.path.exists(cand):
        code = safe_read(cand)
        old = 'torch.load(load_path, map_location="cpu")'
        new = 'torch.load(load_path, map_location="cpu", weights_only=False)'
        if old in code:
            safe_write(cand, code.replace(old, new))
            print('trainer.py 패치 완료')
        else:
            print('trainer.py 이미 패치됨')
        break

print('=' * 50)
print('설치 완료 → [런타임 > 런타임 다시 시작]')

numpy: 1.26.4
DN-Splatter 이미 존재
pyproject.toml 패치 완료
normals_from_pretrain.py 패치 완료
normal_nerfstudio.py 패치 완료
DN-Splatter 설치 완료
DSINE 이미 존재
dn_model.py 패치 완료
regularization_strategy.py 패치 완료
eval_utils.py 패치 완료
trainer.py 이미 패치됨
설치 완료 → [런타임 > 런타임 다시 시작]


In [ ]:
# ============================================================
# Cell 0: Drive 마운트 + GPU 확인
# ============================================================
import subprocess, os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

BASE       = '/content/drive/MyDrive/3dgs_project'
INPUT_DIR  = f'{BASE}/input'
PROC_DIR   = f'{BASE}/processed'
OUTPUT_DIR = f'{BASE}/output'
LOG_DIR    = f'{PROC_DIR}/logs'

for d in [PROC_DIR, OUTPUT_DIR, LOG_DIR,
          f'{OUTPUT_DIR}/logs']:
    os.makedirs(d, exist_ok=True)

gpu_info = subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout
print(gpu_info)

struct_lines = ['=== input/ 구조 ===']
for item in sorted(os.listdir(INPUT_DIR)):
    fp = os.path.join(INPUT_DIR, item)
    if os.path.isdir(fp):
        struct_lines.append(f'  {item}/  ({len(os.listdir(fp))} files)')
    else:
        struct_lines.append(f'  {item}  ({os.path.getsize(fp)/1e6:.1f} MB)')
struct_text = '\n'.join(struct_lines)
print(struct_text)

Path(f'{LOG_DIR}/00_gpu_info.txt').write_text(gpu_info)
Path(f'{LOG_DIR}/00_input_structure.txt').write_text(struct_text)
print('OK: logs/00_*.txt')


Mounted at /content/drive
Sat May 30 11:07:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   33C    P0             53W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+---------------------

In [ ]:
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path

BASE      = '/content/drive/MyDrive/3dgs_project'
INPUT_DIR = f'{BASE}/input'

# ── 1. depth raw 단위 확인 ─────────────────────────────────
depth_files = sorted(Path(f'{INPUT_DIR}/depth').glob('*.png'))
d_raw = np.array(Image.open(depth_files[0]))
print(f'=== Depth Raw ===')
print(f'dtype: {d_raw.dtype}')
print(f'raw min/max: {d_raw.min()} / {d_raw.max()}')
print(f'nonzero mean: {d_raw[d_raw>0].mean():.1f}')
print(f'/1000 시: mean={d_raw[d_raw>0].mean()/1000:.3f}m  max={d_raw.max()/1000:.3f}m')
print(f'/256 시: mean={d_raw[d_raw>0].mean()/256:.3f}m  max={d_raw.max()/256:.3f}m')

# processed depth 확인 (전처리 후 저장된 것)
proc_depth = sorted(Path(f'{BASE}/processed/depth').glob('*.png'))
if proc_depth:
    dp = np.array(Image.open(proc_depth[0]))
    print(f'\n=== Processed Depth ===')
    print(f'dtype: {dp.dtype}')
    print(f'raw min/max: {dp.min()} / {dp.max()}')
    print(f'/1000 시: mean={dp[dp>0].mean()/1000:.3f}m  max={dp.max()/1000:.3f}m')

# ── 2. RGB-Depth 해상도/aspect ratio 확인 ───────────────────
import cv2
cap = cv2.VideoCapture(f'{INPUT_DIR}/rgb.mp4')
W_rgb = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H_rgb = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()
H_d, W_d = d_raw.shape

print(f'\n=== 해상도 ===')
print(f'RGB:   {W_rgb}x{H_rgb}  aspect={W_rgb/H_rgb:.4f}')
print(f'Depth: {W_d}x{H_d}  aspect={W_d/H_d:.4f}')
print(f'aspect 일치: {abs(W_rgb/H_rgb - W_d/H_d) < 0.01}')
print(f'scale: x={W_rgb/W_d:.2f}  y={H_rgb/H_d:.2f}')

# ── 3. arkit_to_nerf 변환 방향 확인 ─────────────────────────
odo = pd.read_csv(f'{INPUT_DIR}/odometry.csv')
odo.columns = [c.strip().lower() for c in odo.columns]

# 첫 프레임 카메라 forward/up 벡터 (변환 전)
import numpy as np
def quat_to_rot(qx,qy,qz,qw):
    return np.array([
        [1-2*(qy**2+qz**2), 2*(qx*qy-qz*qw), 2*(qx*qz+qy*qw)],
        [2*(qx*qy+qz*qw), 1-2*(qx**2+qz**2), 2*(qy*qz-qx*qw)],
        [2*(qx*qz-qy*qw), 2*(qy*qz+qx*qw), 1-2*(qx**2+qy**2)]
    ])

row = odo.iloc[0]
R = quat_to_rot(row['qx'], row['qy'], row['qz'], row['qw'])
m = np.eye(4); m[:3,:3] = R; m[:3,3] = [row['x'], row['y'], row['z']]

print(f'\n=== ARKit 원본 (변환 전) ===')
print(f'forward (Z열): {m[:3,2].round(3)}')
print(f'up (Y열):      {m[:3,1].round(3)}')
print(f'right (X열):   {m[:3,0].round(3)}')

nerf = m @ np.diag([1,-1,-1,1]).astype(float)
print(f'\n=== NeRF 변환 후 ===')
print(f'forward (Z열): {nerf[:3,2].round(3)}')
print(f'up (Y열):      {nerf[:3,1].round(3)}')
print(f'right (X열):   {nerf[:3,0].round(3)}')
print(f'\nup 벡터 Y 성분이 양수여야 정상: {nerf[1,1]:.3f}')

=== Depth Raw ===
dtype: uint16
raw min/max: 1552 / 1899
nonzero mean: 1725.8
/1000 시: mean=1.726m  max=1.899m
/256 시: mean=6.741m  max=7.418m

=== Processed Depth ===
dtype: uint16
raw min/max: 0 / 3979
/1000 시: mean=2.987m  max=3.979m

=== 해상도 ===
RGB:   1920x1440  aspect=1.3333
Depth: 256x192  aspect=1.3333
aspect 일치: True
scale: x=7.50  y=7.50

=== ARKit 원본 (변환 전) ===
forward (Z열): [ 0.997 -0.07  -0.045]
up (Y열):      [-0.07  -0.997 -0.015]
right (X열):   [ 0.044 -0.019  0.999]

=== NeRF 변환 후 ===
forward (Z열): [-0.997  0.07   0.045]
up (Y열):      [0.07  0.997 0.015]
right (X열):   [ 0.044 -0.019  0.999]

up 벡터 Y 성분이 양수여야 정상: 0.997


In [ ]:
# ============================================================
# Cell 1: 기본 패키지 설치
# ============================================================
import subprocess, sys, os

os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']

def run(cmd): subprocess.run(cmd, shell=True)

run('pip install -q opencv-python-headless pandas Pillow tqdm open3d scipy scikit-image')

# DN-Splatter import 확인
r = subprocess.run([sys.executable, '-c',
    'from dn_splatter.dn_config import dn_splatter; print("DN-Splatter OK")'],
    capture_output=True, text=True)
print('DN-Splatter:', 'OK' if 'OK' in r.stdout else '확인 실패 → SETUP 재실행 필요')


DN-Splatter: OK


In [ ]:
# ============================================================
# Cell 2b: CSV 원본 확인
# ============================================================
import pandas as pd
from pathlib import Path

BASE      = '/content/drive/MyDrive/3dgs_project'
INPUT_DIR = f'{BASE}/input'
LOG_DIR   = f'{BASE}/processed/logs'
lines = []

cam = pd.read_csv(f'{INPUT_DIR}/camera_matrix.csv', header=None)
block = f'=== camera_matrix.csv ===\nshape: {cam.shape}\n{cam.to_string()}'
print(block); lines.append(block)

odo = pd.read_csv(f'{INPUT_DIR}/odometry.csv')
block = (f'\n=== odometry.csv ===\nshape: {odo.shape}\n'
         f'컬럼: {list(odo.columns)}\n\n처음 5행:\n{odo.head().to_string()}\n\n'
         f'마지막 3행:\n{odo.tail(3).to_string()}')
print(block); lines.append(block)

d_files = sorted(Path(f'{INPUT_DIR}/depth').glob('*.png'))
c_files = sorted(Path(f'{INPUT_DIR}/confidence').glob('*.png')) \
          if Path(f'{INPUT_DIR}/confidence').exists() else []
block = (f'\n=== depth/ === 총 {len(d_files)} 파일\n'
         f'=== confidence/ === 총 {len(c_files)} 파일')
print(block); lines.append(block)

Path(f'{LOG_DIR}/02b_csv_preview.txt').write_text('\n'.join(lines))
print('\nOK: 타임스탬프 컬럼 이름을 위에서 확인 후 Cell 2에서 설정하세요')


=== camera_matrix.csv ===
shape: (3, 3)
           0          1          2
0  1357.8505     0.0000  956.44410
1     0.0000  1357.8505  728.05426
2     0.0000     0.0000    1.00000

=== odometry.csv ===
shape: (10600, 15)
컬럼: ['timestamp', ' frame', ' x', ' y', ' z', ' qx', ' qy', ' qz', ' qw', ' fx', ' fy', ' cx', ' cy', ' distortion_center_x', ' distortion_center_y']

처음 5행:
      timestamp   frame         x         y         z        qx        qy        qz        qw         fx         fy         cx         cy  distortion_center_x  distortion_center_y
0  1.863875e+06       0  0.196281  0.162400  4.244051  0.722270 -0.030735  0.690673  0.018747  1345.6678  1345.6678  956.90240  728.13050                                          
1  1.863875e+06       1  0.194361  0.163189  4.242169  0.722769 -0.029032  0.690181  0.020301  1345.6827  1345.6827  956.87317  728.12744                                          
2  1.863875e+06       2  0.185307  0.163349  4.241806  0.722043 -0.027158  0.6910

In [ ]:
# ▶ 기존 전처리 결과 초기화 (새 영상 처리 전 필수)
import shutil
from pathlib import Path

BASE      = '/content/drive/MyDrive/3dgs_project'
PROC_DIR  = f'{BASE}/processed'

for d in ['images', 'depth']:
    p = Path(PROC_DIR) / d
    if p.exists():
        shutil.rmtree(p)
        print(f'{d}/ 삭제 완료')
    p.mkdir(exist_ok=True)

# transforms.json, sparse_pointcloud.ply 삭제
for f in ['transforms.json', 'sparse_pointcloud.ply']:
    fp = Path(PROC_DIR) / f
    if fp.exists():
        fp.unlink()
        print(f'{f} 삭제 완료')

images/ 삭제 완료
depth/ 삭제 완료


In [ ]:
# ============================================================
# Cell 2: StrayScanner 전처리 (확정판)
# ▶ frame 컬럼으로 직접 포즈 매핑 (타임스탬프 방식 폐기)
# ▶ odometry의 fx,fy,cx,cy 프레임별 사용
# ▶ depth 해상도(192×256) 별도 intrinsics로 back-projection
# ▶ CONF_THRESH=2, JUMP_THRESH=0.5
# ============================================================
import cv2, json, os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from PIL import Image

TARGET_FPS  = 5
CONF_THRESH = 2
JUMP_THRESH = 0.5

BASE      = '/content/drive/MyDrive/3dgs_project'
INPUT_DIR = f'{BASE}/input'
PROC_DIR  = f'{BASE}/processed'
LOG_DIR   = f'{PROC_DIR}/logs'
IMG_DIR   = Path(PROC_DIR) / 'images'
DEPTH_DIR = Path(PROC_DIR) / 'depth'
IMG_DIR.mkdir(exist_ok=True)
DEPTH_DIR.mkdir(exist_ok=True)

log_lines = []
def log(msg): print(msg); log_lines.append(str(msg))

# 1. RGB 추출
cap      = cv2.VideoCapture(f'{INPUT_DIR}/rgb.mp4')
src_fps  = cap.get(cv2.CAP_PROP_FPS)
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
log(f'원본 영상: {n_frames} frames @ {src_fps:.1f}fps  ({W}x{H})')

stride = max(1, round(src_fps / TARGET_FPS))
frame_indices = list(range(0, n_frames, stride))
log(f'stride={stride}  ->  추출 예정: {len(frame_indices)} frames')

saved_frame_ids = []
for idx in tqdm(frame_indices, desc='RGB 추출'):
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ret, frame = cap.read()
    if not ret: continue
    cv2.imwrite(str(IMG_DIR / f'frame_{idx:06d}.jpg'), frame,
                [cv2.IMWRITE_JPEG_QUALITY, 95])
    saved_frame_ids.append(idx)
cap.release()
log(f'저장된 RGB 프레임: {len(saved_frame_ids)}')

# 2. Depth + Confidence Masking
depth_index = {int(f.stem): f for f in sorted((Path(INPUT_DIR)/'depth').glob('*.png'))}
conf_index  = {int(f.stem): f for f in sorted((Path(INPUT_DIR)/'confidence').glob('*.png'))} \
              if (Path(INPUT_DIR)/'confidence').exists() else {}
log(f'Depth: {len(depth_index)}  Confidence: {len(conf_index)}')

saved_depth_ids, mask_ratios = [], []
for idx in tqdm(saved_frame_ids, desc='Depth 처리'):
    d_idx = idx if idx in depth_index \
            else min(depth_index.keys(), key=lambda x: abs(x-idx))
    depth_img = np.array(Image.open(depth_index[d_idx]))
    if d_idx in conf_index:
        conf_img = np.array(Image.open(conf_index[d_idx]))
        mask = conf_img < CONF_THRESH
        mask_ratios.append(mask.mean())
        depth_img = depth_img.copy(); depth_img[mask] = 0
    Image.fromarray(depth_img).save(str(DEPTH_DIR / f'frame_{idx:06d}.png'))
    saved_depth_ids.append(idx)
if mask_ratios:
    log(f'Confidence masking 평균 비율: {np.mean(mask_ratios)*100:.1f}%')

# 3. Odometry 파싱
odo = pd.read_csv(f'{INPUT_DIR}/odometry.csv')
odo.columns = [c.strip().lower() for c in odo.columns]
cols = list(odo.columns)
log(f'Odometry 컬럼: {cols}  행 수: {len(odo)}')

def get_col(candidates):
    for c in candidates:
        if c in cols: return c
    raise KeyError(f'{candidates} 없음. 컬럼: {cols}')

col_x  = get_col(['x','tx','pos_x','position_x'])
col_y  = get_col(['y','ty','pos_y','position_y'])
col_z  = get_col(['z','tz','pos_z','position_z'])
col_qx = get_col(['qx','rot_x','rotation_x'])
col_qy = get_col(['qy','rot_y','rotation_y'])
col_qz = get_col(['qz','rot_z','rotation_z'])
col_qw = get_col(['qw','rot_w','rotation_w'])

# ▶ frame 컬럼으로 직접 매핑
has_frame_col = 'frame' in cols
if has_frame_col:
    odo_by_frame = {int(row['frame']): row for _, row in odo.iterrows()}
    log(f'frame 컬럼 매핑  (범위: {int(odo["frame"].min())} ~ {int(odo["frame"].max())})')
else:
    odo_by_frame = {i: row for i, (_, row) in enumerate(odo.iterrows())}
    log('frame 컬럼 없음 → 인덱스 기반 매핑')

# ▶ odometry에서 fx,fy,cx,cy 읽기
has_intrinsics = all(c in cols for c in ['fx','fy','cx','cy'])
if has_intrinsics:
    fx_mean = float(odo['fx'].mean())
    fy_mean = float(odo['fy'].mean())
    cx_mean = float(odo['cx'].mean())
    cy_mean = float(odo['cy'].mean())
    log(f'odometry 내부 파라미터: fx={fx_mean:.2f} fy={fy_mean:.2f} cx={cx_mean:.2f} cy={cy_mean:.2f}')
else:
    K_data = pd.read_csv(f'{INPUT_DIR}/camera_matrix.csv', header=None).values.flatten()
    if len(K_data) == 9:
        K = K_data.reshape(3,3); fx_mean,fy_mean,cx_mean,cy_mean = K[0,0],K[1,1],K[0,2],K[1,2]
    else:
        fx_mean,fy_mean,cx_mean,cy_mean = K_data
    log(f'camera_matrix.csv: fx={fx_mean:.2f} fy={fy_mean:.2f}')

def quat_to_rot(qx,qy,qz,qw):
    return np.array([
        [1-2*(qy**2+qz**2), 2*(qx*qy-qz*qw), 2*(qx*qz+qy*qw)],
        [2*(qx*qy+qz*qw), 1-2*(qx**2+qz**2), 2*(qy*qz-qx*qw)],
        [2*(qx*qz-qy*qw), 2*(qy*qz+qx*qw), 1-2*(qx**2+qy**2)]
    ])

def arkit_to_nerf(c2w):
    return c2w @ np.diag([1,-1,-1,1]).astype(float)

def row_to_c2w(row):
    R = quat_to_rot(row[col_qx], row[col_qy], row[col_qz], row[col_qw])
    m = np.eye(4); m[:3,:3] = R; m[:3,3] = [row[col_x], row[col_y], row[col_z]]
    return m

# 포즈 점프 필터링
all_frames_sorted = sorted(odo_by_frame.keys())
positions_odo = np.array([
    [odo_by_frame[k][col_x], odo_by_frame[k][col_y], odo_by_frame[k][col_z]]
    for k in all_frames_sorted
])
diffs_odo = np.linalg.norm(np.diff(positions_odo, axis=0), axis=1)
jump_frames = set(all_frames_sorted[i+1] for i, d in enumerate(diffs_odo) if d > JUMP_THRESH)
log(f'포즈 점프 감지: {len(jump_frames)}건 (>{JUMP_THRESH}m)')

# 4. transforms.json 생성
frames = []
for idx in saved_frame_ids:
    if idx in odo_by_frame:
        row = odo_by_frame[idx]
    else:
        closest = min(odo_by_frame.keys(), key=lambda x: abs(x - idx))
        row = odo_by_frame[closest]

    c2w  = arkit_to_nerf(row_to_c2w(row))
    fl_x = float(row['fx']) if has_intrinsics else fx_mean
    fl_y = float(row['fy']) if has_intrinsics else fy_mean
    c_x  = float(row['cx']) if has_intrinsics else cx_mean
    c_y  = float(row['cy']) if has_intrinsics else cy_mean

    entry = {'file_path': f'images/frame_{idx:06d}.jpg',
             'transform_matrix': c2w.tolist(),
             'fl_x': fl_x, 'fl_y': fl_y, 'cx': c_x, 'cy': c_y}

    depth_path = DEPTH_DIR / f'frame_{idx:06d}.png'
    if depth_path.exists():
        if idx in jump_frames:
            depth_path.unlink(missing_ok=True)
        else:
            entry['depth_file_path'] = f'depth/frame_{idx:06d}.png'
    frames.append(entry)

transforms = {'fl_x': fx_mean, 'fl_y': fy_mean, 'cx': cx_mean, 'cy': cy_mean,
              'w': W, 'h': H, 'camera_model': 'PINHOLE', 'frames': frames}
with open(f'{PROC_DIR}/transforms.json', 'w') as f:
    json.dump(transforms, f, indent=2)
n_d = sum(1 for fr in frames if 'depth_file_path' in fr)
log(f'transforms.json: {len(frames)} frames (depth 포함: {n_d})')

# 5. 포즈 범위 확인
positions = np.array([np.array(fr['transform_matrix'])[:3,3] for fr in frames])
for i, name in enumerate(['X','Y','Z']):
    log(f'{name}: [{positions[:,i].min():.2f}, {positions[:,i].max():.2f}]  range={np.ptp(positions[:,i]):.2f}m')

# 6. 시각화
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (xi,zi,title) in zip(axes, [(0,2,'Top (X-Z)'),(0,1,'Front (X-Y)'),(2,1,'Side (Z-Y)')]):
    ax.plot(positions[:,xi], positions[:,zi], 'b-o', ms=2, lw=0.8)
    ax.set_title(title); ax.set_aspect('equal')
plt.suptitle(f'Camera Trajectory ({len(frames)} frames)', fontsize=13)
plt.tight_layout()
plt.savefig(f'{LOG_DIR}/02_trajectory.png', dpi=120); plt.close()

Path(f'{LOG_DIR}/02_preprocessing_summary.txt').write_text('\n'.join(log_lines))
print('\n전처리 완료')


원본 영상: 4485 frames @ 30.0fps  (1920x1440)
stride=6  ->  추출 예정: 748 frames


RGB 추출: 100%|██████████| 748/748 [03:00<00:00,  4.15it/s]


저장된 RGB 프레임: 748
Depth: 4485  Confidence: 4485


Depth 처리: 100%|██████████| 748/748 [16:26<00:00,  1.32s/it]


Confidence masking 평균 비율: 26.1%
Odometry 컬럼: ['timestamp', 'frame', 'x', 'y', 'z', 'qx', 'qy', 'qz', 'qw', 'fx', 'fy', 'cx', 'cy', 'distortion_center_x', 'distortion_center_y']  행 수: 4485
frame 컬럼 매핑  (범위: 0 ~ 4484)
odometry 내부 파라미터: fx=1344.56 fy=1344.56 cx=955.07 cy=727.65
포즈 점프 감지: 0건 (>0.5m)
transforms.json: 748 frames (depth 포함: 748)
X: [-32.84, 15.30]  range=48.13m
Y: [0.19, 0.53]  range=0.33m
Z: [-23.80, 30.14]  range=53.95m

전처리 완료


In [ ]:
# ============================================================
# Cell 2: StrayScanner 전처리 (확정판 + 실험 B)
# ▶ TARGET_FPS=10, JPEG_QUALITY=100
# ▶ blur 프레임 제거 없음 (로그만 기록)
# ▶ sharpen 기본 OFF
# ============================================================
import cv2, json, os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from PIL import Image
import shutil

TARGET_FPS    = 10
CONF_THRESH   = 2
JUMP_THRESH   = 0.5
JPEG_QUALITY  = 100
USE_SHARPEN   = False   # 포스터 디테일 실험 시 True로 변경

BASE      = '/content/drive/MyDrive/3dgs_project'
INPUT_DIR = f'{BASE}/input'
PROC_DIR  = f'{BASE}/processed'
LOG_DIR   = f'{PROC_DIR}/logs'
IMG_DIR   = Path(PROC_DIR) / 'images'
DEPTH_DIR = Path(PROC_DIR) / 'depth'

# 기존 전처리 결과 초기화
for d in [IMG_DIR, DEPTH_DIR]:
    if d.exists(): shutil.rmtree(d)
    d.mkdir(exist_ok=True)
for f in ['transforms.json', 'sparse_pointcloud.ply']:
    fp = Path(PROC_DIR) / f
    if fp.exists(): fp.unlink(); print(f'{f} 삭제')

log_lines = []
def log(msg): print(msg); log_lines.append(str(msg))

def blur_score(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return cv2.Laplacian(gray, cv2.CV_64F).var()

def sharpen(img):
    gaussian = cv2.GaussianBlur(img, (0, 0), 2.0)
    return cv2.addWeighted(img, 1.5, gaussian, -0.5, 0)

# 1. RGB 추출
cap      = cv2.VideoCapture(f'{INPUT_DIR}/rgb.mp4')
src_fps  = cap.get(cv2.CAP_PROP_FPS)
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
log(f'원본 영상: {n_frames} frames @ {src_fps:.1f}fps  ({W}x{H})')

stride = max(1, round(src_fps / TARGET_FPS))
frame_indices = list(range(0, n_frames, stride))
log(f'stride={stride}  ->  추출 예정: {len(frame_indices)} frames')
log(f'USE_SHARPEN={USE_SHARPEN}  JPEG_QUALITY={JPEG_QUALITY}')

saved_frame_ids = []
blur_scores = []

for idx in tqdm(frame_indices, desc='RGB 추출'):
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ret, frame = cap.read()
    if not ret: continue

    score = blur_score(frame)
    blur_scores.append((idx, score))

    if USE_SHARPEN:
        frame = sharpen(frame)

    cv2.imwrite(str(IMG_DIR / f'frame_{idx:06d}.jpg'), frame,
                [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
    saved_frame_ids.append(idx)

cap.release()
log(f'저장된 RGB 프레임: {len(saved_frame_ids)}')

# blur score 로그 저장
scores_arr = np.array([s for _, s in blur_scores])
log(f'blur score: mean={scores_arr.mean():.1f}  min={scores_arr.min():.1f}  max={scores_arr.max():.1f}')
log(f'blur score <30 프레임: {(scores_arr<30).sum()}건  <50: {(scores_arr<50).sum()}건')

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot([idx for idx, _ in blur_scores], scores_arr, lw=0.5)
ax.axhline(30, color='orange', ls='--', label='30')
ax.axhline(50, color='red', ls='--', label='50')
ax.set_title('Blur Score (프레임별)')
ax.set_xlabel('frame index'); ax.legend()
plt.tight_layout()
plt.savefig(f'{LOG_DIR}/02_blur_scores.png', dpi=120); plt.close()

# 2. Depth + Confidence Masking
depth_index = {int(f.stem): f for f in sorted((Path(INPUT_DIR)/'depth').glob('*.png'))}
conf_index  = {int(f.stem): f for f in sorted((Path(INPUT_DIR)/'confidence').glob('*.png'))} \
              if (Path(INPUT_DIR)/'confidence').exists() else {}
log(f'Depth: {len(depth_index)}  Confidence: {len(conf_index)}')

saved_depth_ids, mask_ratios = [], []
for idx in tqdm(saved_frame_ids, desc='Depth 처리'):
    d_idx = idx if idx in depth_index \
            else min(depth_index.keys(), key=lambda x: abs(x-idx))
    depth_img = np.array(Image.open(depth_index[d_idx]))
    if d_idx in conf_index:
        conf_img = np.array(Image.open(conf_index[d_idx]))
        mask = conf_img < CONF_THRESH
        mask_ratios.append(mask.mean())
        depth_img = depth_img.copy(); depth_img[mask] = 0
    Image.fromarray(depth_img).save(str(DEPTH_DIR / f'frame_{idx:06d}.png'))
    saved_depth_ids.append(idx)
if mask_ratios:
    log(f'Confidence masking 평균 비율: {np.mean(mask_ratios)*100:.1f}%')

# 3. Odometry 파싱
odo = pd.read_csv(f'{INPUT_DIR}/odometry.csv')
odo.columns = [c.strip().lower() for c in odo.columns]
cols = list(odo.columns)
log(f'Odometry 컬럼: {cols}  행 수: {len(odo)}')

def get_col(candidates):
    for c in candidates:
        if c in cols: return c
    raise KeyError(f'{candidates} 없음. 컬럼: {cols}')

col_x  = get_col(['x','tx','pos_x','position_x'])
col_y  = get_col(['y','ty','pos_y','position_y'])
col_z  = get_col(['z','tz','pos_z','position_z'])
col_qx = get_col(['qx','rot_x','rotation_x'])
col_qy = get_col(['qy','rot_y','rotation_y'])
col_qz = get_col(['qz','rot_z','rotation_z'])
col_qw = get_col(['qw','rot_w','rotation_w'])

has_frame_col = 'frame' in cols
if has_frame_col:
    odo_by_frame = {int(row['frame']): row for _, row in odo.iterrows()}
    log(f'frame 컬럼 매핑  (범위: {int(odo["frame"].min())} ~ {int(odo["frame"].max())})')
else:
    odo_by_frame = {i: row for i, (_, row) in enumerate(odo.iterrows())}
    log('frame 컬럼 없음 → 인덱스 기반 매핑')

has_intrinsics = all(c in cols for c in ['fx','fy','cx','cy'])
if has_intrinsics:
    fx_mean = float(odo['fx'].mean())
    fy_mean = float(odo['fy'].mean())
    cx_mean = float(odo['cx'].mean())
    cy_mean = float(odo['cy'].mean())
    log(f'odometry 내부 파라미터: fx={fx_mean:.2f} fy={fy_mean:.2f}')
else:
    K_data = pd.read_csv(f'{INPUT_DIR}/camera_matrix.csv', header=None).values.flatten()
    if len(K_data) == 9:
        K = K_data.reshape(3,3); fx_mean,fy_mean,cx_mean,cy_mean = K[0,0],K[1,1],K[0,2],K[1,2]
    else:
        fx_mean,fy_mean,cx_mean,cy_mean = K_data
    log(f'camera_matrix.csv: fx={fx_mean:.2f} fy={fy_mean:.2f}')

def quat_to_rot(qx,qy,qz,qw):
    return np.array([
        [1-2*(qy**2+qz**2), 2*(qx*qy-qz*qw), 2*(qx*qz+qy*qw)],
        [2*(qx*qy+qz*qw), 1-2*(qx**2+qz**2), 2*(qy*qz-qx*qw)],
        [2*(qx*qz-qy*qw), 2*(qy*qz+qx*qw), 1-2*(qx**2+qy**2)]
    ])

def arkit_to_nerf(c2w):
    return c2w @ np.diag([1,-1,-1,1]).astype(float)

def row_to_c2w(row):
    R = quat_to_rot(row[col_qx], row[col_qy], row[col_qz], row[col_qw])
    m = np.eye(4); m[:3,:3] = R; m[:3,3] = [row[col_x], row[col_y], row[col_z]]
    return m

# 포즈 점프 필터링
all_frames_sorted = sorted(odo_by_frame.keys())
positions_odo = np.array([
    [odo_by_frame[k][col_x], odo_by_frame[k][col_y], odo_by_frame[k][col_z]]
    for k in all_frames_sorted
])
diffs_odo = np.linalg.norm(np.diff(positions_odo, axis=0), axis=1)
jump_frames = set(all_frames_sorted[i+1] for i, d in enumerate(diffs_odo) if d > JUMP_THRESH)
log(f'포즈 점프 감지: {len(jump_frames)}건 (>{JUMP_THRESH}m)')

# 4. transforms.json 생성
frames = []
for idx in saved_frame_ids:
    if idx in odo_by_frame:
        row = odo_by_frame[idx]
    else:
        closest = min(odo_by_frame.keys(), key=lambda x: abs(x - idx))
        row = odo_by_frame[closest]

    c2w  = arkit_to_nerf(row_to_c2w(row))
    fl_x = float(row['fx']) if has_intrinsics else fx_mean
    fl_y = float(row['fy']) if has_intrinsics else fy_mean
    c_x  = float(row['cx']) if has_intrinsics else cx_mean
    c_y  = float(row['cy']) if has_intrinsics else cy_mean

    entry = {'file_path': f'images/frame_{idx:06d}.jpg',
             'transform_matrix': c2w.tolist(),
             'fl_x': fl_x, 'fl_y': fl_y, 'cx': c_x, 'cy': c_y}

    depth_path = DEPTH_DIR / f'frame_{idx:06d}.png'
    if depth_path.exists():
        if idx in jump_frames:
            depth_path.unlink(missing_ok=True)
        else:
            entry['depth_file_path'] = f'depth/frame_{idx:06d}.png'
    frames.append(entry)

transforms = {'fl_x': fx_mean, 'fl_y': fy_mean, 'cx': cx_mean, 'cy': cy_mean,
              'w': W, 'h': H, 'camera_model': 'PINHOLE', 'frames': frames}
with open(f'{PROC_DIR}/transforms.json', 'w') as f:
    json.dump(transforms, f, indent=2)
n_d = sum(1 for fr in frames if 'depth_file_path' in fr)
log(f'transforms.json: {len(frames)} frames (depth 포함: {n_d})')

# 5. 포즈 범위 확인
positions = np.array([np.array(fr['transform_matrix'])[:3,3] for fr in frames])
for i, name in enumerate(['X','Y','Z']):
    log(f'{name}: [{positions[:,i].min():.2f}, {positions[:,i].max():.2f}]  range={np.ptp(positions[:,i]):.2f}m')

# 6. 시각화
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (xi,zi,title) in zip(axes, [(0,2,'Top (X-Z)'),(0,1,'Front (X-Y)'),(2,1,'Side (Z-Y)')]):
    ax.plot(positions[:,xi], positions[:,zi], 'b-o', ms=2, lw=0.8)
    ax.set_title(title); ax.set_aspect('equal')
plt.suptitle(f'Trajectory ({len(frames)} frames)', fontsize=13)
plt.tight_layout()
plt.savefig(f'{LOG_DIR}/02_trajectory.png', dpi=120); plt.close()

Path(f'{LOG_DIR}/02_preprocessing_summary.txt').write_text('\n'.join(log_lines))
print('\n전처리 완료')
print(f'\n→ logs/02_blur_scores.png 확인 후 blur 제거 여부 결정')

transforms.json 삭제
sparse_pointcloud.ply 삭제
원본 영상: 2860 frames @ 30.0fps  (1920x1440)
stride=3  ->  추출 예정: 954 frames
USE_SHARPEN=False  JPEG_QUALITY=100


RGB 추출: 100%|██████████| 954/954 [04:03<00:00,  3.91it/s]
/tmp/ipykernel_10775/4046349348.py:95: UserWarning: Glyph 54532 (\N{HANGUL SYLLABLE PEU}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_10775/4046349348.py:95: UserWarning: Glyph 47112 (\N{HANGUL SYLLABLE RE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_10775/4046349348.py:95: UserWarning: Glyph 51076 (\N{HANGUL SYLLABLE IM}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_10775/4046349348.py:95: UserWarning: Glyph 48324 (\N{HANGUL SYLLABLE BYEOL}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_10775/4046349348.py:96: UserWarning: Glyph 54532 (\N{HANGUL SYLLABLE PEU}) missing from font(s) DejaVu Sans.
  plt.savefig(f'{LOG_DIR}/02_blur_scores.png', dpi=120); plt.close()
/tmp/ipykernel_10775/4046349348.py:96: UserWarning: Glyph 47112 (\N{HANGUL SYLLABLE RE}) missing from font(s) DejaVu Sans.
  plt.savefig(f'{LOG_DIR}/02_blur_scores.png',

저장된 RGB 프레임: 953
blur score: mean=22.6  min=0.5  max=113.6
blur score <30 프레임: 720건  <50: 888건
Depth: 2860  Confidence: 2860


Depth 처리: 100%|██████████| 953/953 [11:17<00:00,  1.41it/s]


Confidence masking 평균 비율: 22.2%
Odometry 컬럼: ['timestamp', 'frame', 'x', 'y', 'z', 'qx', 'qy', 'qz', 'qw', 'fx', 'fy', 'cx', 'cy', 'distortion_center_x', 'distortion_center_y']  행 수: 2860
frame 컬럼 매핑  (범위: 0 ~ 2859)
odometry 내부 파라미터: fx=1340.92 fy=1340.92
포즈 점프 감지: 2건 (>0.5m)
transforms.json: 953 frames (depth 포함: 952)
X: [-32.65, 0.63]  range=33.28m
Y: [-0.19, 0.23]  range=0.42m
Z: [-6.16, 21.23]  range=27.39m

전처리 완료

→ logs/02_blur_scores.png 확인 후 blur 제거 여부 결정


In [ ]:
# Cell 2c: transforms.json 검증
import json, numpy as np, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

BASE    = '/content/drive/MyDrive/3dgs_project'
PROC    = f'{BASE}/processed'
LOG_DIR = f'{PROC}/logs'

with open(f'{PROC}/transforms.json') as f:
    tf = json.load(f)

lines = []
def log(msg): print(msg); lines.append(str(msg))

log(f'총 프레임: {len(tf["frames"])}')
log(f'해상도: {tf["w"]}x{tf["h"]}')
log(f'fl_x={tf["fl_x"]:.2f}  fl_y={tf["fl_y"]:.2f}  cx={tf["cx"]:.2f}  cy={tf["cy"]:.2f}')
n_d = sum(1 for f in tf['frames'] if 'depth_file_path' in f)
log(f'depth 포함: {n_d}/{len(tf["frames"])}')

positions = np.array([np.array(f['transform_matrix'])[:3,3] for f in tf['frames']])
log('\n포즈 범위:')
for i, name in enumerate(['X','Y','Z']):
    log(f'  {name}: [{positions[:,i].min():.3f}, {positions[:,i].max():.3f}]  '
        f'range={np.ptp(positions[:,i]):.3f}m')

diffs = np.linalg.norm(np.diff(positions, axis=0), axis=1)
log(f'\n인접 이동거리: mean={diffs.mean():.4f}m  max={diffs.max():.4f}m')
log(f'점프(>0.5m): {(diffs>0.5).sum()}건')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(diffs, bins=50, color='steelblue', edgecolor='white')
axes[0].axvline(diffs.mean(), color='red', ls='--', label=f'mean={diffs.mean():.3f}m')
axes[0].set_title('인접 이동거리 분포'); axes[0].legend()
sc = axes[1].scatter(positions[:,0], positions[:,2],
                     c=np.arange(len(positions)), cmap='viridis', s=10)
plt.colorbar(sc, ax=axes[1], label='frame index')
axes[1].set_title('Trajectory (색상=시간순)'); axes[1].set_aspect('equal')
plt.tight_layout()
plt.savefig(f'{LOG_DIR}/02c_pose_distribution.png', dpi=120); plt.close()

Path(f'{LOG_DIR}/02c_transforms_check.txt').write_text('\n'.join(lines))
print('OK')


총 프레임: 477
해상도: 1920x1440
fl_x=1340.92  fl_y=1340.92  cx=956.29  cy=727.81
depth 포함: 477/477

포즈 범위:
  X: [-32.644, 0.629]  range=33.274m
  Y: [-0.188, 0.228]  range=0.417m
  Z: [-6.158, 21.232]  range=27.390m

인접 이동거리: mean=0.2269m  max=2.6868m
점프(>0.5m): 1건


/tmp/ipykernel_3422/2026076961.py:41: UserWarning: Glyph 51064 (\N{HANGUL SYLLABLE IN}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3422/2026076961.py:41: UserWarning: Glyph 51217 (\N{HANGUL SYLLABLE JEOB}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3422/2026076961.py:41: UserWarning: Glyph 51060 (\N{HANGUL SYLLABLE I}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3422/2026076961.py:41: UserWarning: Glyph 46041 (\N{HANGUL SYLLABLE DONG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3422/2026076961.py:41: UserWarning: Glyph 44144 (\N{HANGUL SYLLABLE GEO}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3422/2026076961.py:41: UserWarning: Glyph 47532 (\N{HANGUL SYLLABLE RI}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3422/2026076961.py:41: UserWarning: Glyph 48516 (\N{HANGUL SYLLABLE BUN}) missing from font(s) DejaVu Sans.
  plt.tight_lay

OK


In [ ]:
# ============================================================
# Cell 2e: LiDAR depth → 포인트클라우드 생성 (확정판)
# ▶ depth 해상도(192×256) 기준 intrinsics 적용
# ▶ RGB 해상도(1920×1440)와 7.5배 차이 → scale 보정 필수
# ============================================================
import numpy as np, json, struct, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from tqdm import tqdm

BASE     = '/content/drive/MyDrive/3dgs_project'
PROC_DIR = f'{BASE}/processed'
LOG_DIR  = f'{PROC_DIR}/logs'

with open(f'{PROC_DIR}/transforms.json') as f:
    tf = json.load(f)

# RGB 해상도 기준 내부 파라미터
fx_rgb = tf['fl_x']; fy_rgb = tf['fl_y']
cx_rgb = tf['cx'];   cy_rgb = tf['cy']
W_rgb  = tf['w'];    H_rgb  = tf['h']

# depth 해상도 확인 및 스케일 계산
sample_depth_path = Path(PROC_DIR) / tf['frames'][0]['depth_file_path']
d_sample = np.array(Image.open(sample_depth_path))
H_d, W_d = d_sample.shape
scale_x = W_d / W_rgb; scale_y = H_d / H_rgb
fx_d = fx_rgb * scale_x; fy_d = fy_rgb * scale_y
cx_d = cx_rgb * scale_x; cy_d = cy_rgb * scale_y
print(f'RGB: {W_rgb}x{H_rgb}  Depth: {W_d}x{H_d}  scale: {scale_x:.3f}')
print(f'Depth intrinsics: fx={fx_d:.2f} fy={fy_d:.2f} cx={cx_d:.2f} cy={cy_d:.2f}')

points_xyz, points_rgb = [], []

for frame in tqdm(tf['frames'][::10], desc='포인트클라우드 생성'):
    if 'depth_file_path' not in frame: continue
    depth = np.array(Image.open(f"{PROC_DIR}/{frame['depth_file_path']}")).astype(np.float32) / 1000.0
    img   = np.array(Image.open(f"{PROC_DIR}/{frame['file_path']}"))
    img_d = np.array(Image.fromarray(img).resize((W_d, H_d), Image.BILINEAR))
    c2w   = np.array(frame['transform_matrix'])
    valid = (depth > 0.1) & (depth < 10.0)
    ys, xs = np.where(valid)
    if len(ys) == 0: continue
    sel = np.random.choice(len(ys), min(2000, len(ys)), replace=False)
    ys, xs = ys[sel], xs[sel]
    zs = depth[ys, xs]
    pts_cam = np.stack([(xs-cx_d)*zs/fx_d, (ys-cy_d)*zs/fy_d, zs, np.ones_like(zs)], axis=1)
    pts_world = (c2w @ pts_cam.T).T[:, :3]
    points_xyz.append(pts_world)
    points_rgb.append(img_d[ys, xs, :3])

points_xyz = np.concatenate(points_xyz, axis=0)
points_rgb = np.concatenate(points_rgb, axis=0)
print(f'총 포인트 수: {len(points_xyz):,}')

ply_path = f'{PROC_DIR}/sparse_pointcloud.ply'
with open(ply_path, 'wb') as f:
    header = (f'ply\nformat binary_little_endian 1.0\n'
              f'element vertex {len(points_xyz)}\n'
              f'property float x\nproperty float y\nproperty float z\n'
              f'property uchar red\nproperty uchar green\nproperty uchar blue\n'
              f'end_header\n').encode()
    f.write(header)
    for p, c in zip(points_xyz, points_rgb):
        f.write(struct.pack('fffBBB', p[0],p[1],p[2], c[0],c[1],c[2]))

tf['ply_file_path'] = 'sparse_pointcloud.ply'
with open(f'{PROC_DIR}/transforms.json', 'w') as f:
    json.dump(tf, f, indent=2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.scatter(points_xyz[::10,0], points_xyz[::10,2], s=0.5,
            c=points_rgb[::10]/255.0, linewidths=0)
ax1.set_title('Top view (X-Z)'); ax1.set_aspect('equal')
ax2.scatter(points_xyz[::10,0], points_xyz[::10,1], s=0.5,
            c=points_rgb[::10]/255.0, linewidths=0)
ax2.set_title('Front view (X-Y)'); ax2.set_aspect('equal')
plt.tight_layout()
plt.savefig(f'{LOG_DIR}/02e_pointcloud.png', dpi=120); plt.close()
print('OK: sparse_pointcloud.ply')


RGB: 1920x1440  Depth: 256x192  scale: 0.133
Depth intrinsics: fx=179.27 fy=179.27 cx=127.34 cy=97.02


포인트클라우드 생성: 100%|██████████| 75/75 [00:03<00:00, 22.40it/s]


총 포인트 수: 150,000
OK: sparse_pointcloud.ply


In [ ]:
# ============================================================
# Cell 3: AGS-Mesh 학습 (확정판)
# ▶ scene-scale=75  auto-scale-poses=False  load-3D-points=True
# ▶ max-num-iterations=30000
# ============================================================
import subprocess, os, re, sys, time
from pathlib import Path
from IPython.display import clear_output

os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']

BASE       = '/content/drive/MyDrive/3dgs_project'
PROC_DIR   = f'{BASE}/processed'
OUTPUT_DIR = f'{BASE}/output'
os.makedirs(f'{OUTPUT_DIR}/logs', exist_ok=True)

EXP_NAME  = f'indoor_scan_{time.strftime("%m%d_%H%M%S")}'
MAX_ITER  = 30000
LOG_EVERY = 50

cmd = [
    sys.executable, '-m', 'nerfstudio.scripts.train',
    'ags-mesh',
    '--output-dir', OUTPUT_DIR,
    '--experiment-name', EXP_NAME,
    '--max-num-iterations', str(MAX_ITER),
    '--vis', 'tensorboard',
    '--pipeline.model.use-depth-loss', 'True',
    '--pipeline.model.depth-lambda', '0.2',
    '--pipeline.model.use-normal-loss', 'True',
    '--pipeline.model.use-normal-tv-loss', 'True',
    '--pipeline.model.normal-supervision', 'depth',
    'normal-nerfstudio',
    '--data', PROC_DIR,
    '--load-3D-points', 'True',        # 포인트클라우드 초기값
    '--load-normals', 'False',
    '--load-depth-confidence-masks', 'False',
    '--downscale-factor', '1',
    '--scene-scale', '75.0',           # 복도 56m 커버
    '--auto-scale-poses', 'False',     # nerfstudio 압축 방지
]

print(f'실험: {EXP_NAME}')
pat_step = re.compile(r'^\s*(\d+)\s*\(')
pat_loss = re.compile(r'loss[:\s]+([\d\.eE+\-]+)', re.I)
pat_psnr = re.compile(r'psnr[:\s]+([\d\.]+)', re.I)
last_step, last_loss, last_psnr = 0, '-', '-'
recent_lines = []
start_time = time.time()

def render_status():
    elapsed = time.time() - start_time
    pct = last_step / MAX_ITER * 100
    bar = '#' * int(pct/5) + '.' * (20 - int(pct/5))
    eta_str = time.strftime('%H:%M:%S', time.gmtime(
        elapsed / last_step * (MAX_ITER - last_step))) if last_step > 0 else '--:--:--'
    clear_output(wait=True)
    print(f'--- AGS-Mesh 학습 ---  실험: {EXP_NAME}')
    print(f'  진행: [{bar}] {pct:5.1f}%  ({last_step:>6}/{MAX_ITER})')
    print(f'  경과: {time.strftime("%H:%M:%S", time.gmtime(elapsed))}   ETA: {eta_str}')
    print(f'  Loss: {last_loss:<12}  PSNR: {last_psnr}')
    print('최근 로그:')
    for l in recent_lines[-10:]: print(' ', l)

with open(f'{OUTPUT_DIR}/logs/training_log.txt', 'w') as logf:
    proc = subprocess.Popen(
        cmd, cwd='/content/dn-splatter',
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1)
    for raw_line in proc.stdout:
        for line in raw_line.replace('\r', '\n').split('\n'):
            line = line.strip()
            if not line: continue
            logf.write(line + '\n'); logf.flush()
            recent_lines.append(line)
            if len(recent_lines) > 100: recent_lines.pop(0)
            m = pat_step.search(line)
            if m: last_step = int(m.group(1))
            m = pat_loss.search(line)
            if m: last_loss = m.group(1)
            m = pat_psnr.search(line)
            if m: last_psnr = m.group(1)
            if last_step > 0 and last_step % LOG_EVERY == 0:
                render_status()
    proc.wait()

last_step = MAX_ITER; render_status()
print(f'\n소요: {(time.time()-start_time)/60:.1f}분  Return code: {proc.returncode}')

if proc.returncode == 0:
    import torch, numpy as np
    ckpts = sorted([c for c in Path(OUTPUT_DIR).rglob('*.ckpt')
                    if EXP_NAME in str(c)], key=lambda x: x.stat().st_mtime)
    state = torch.load(str(ckpts[-1]), map_location='cpu', weights_only=False)
    means = state['pipeline']['_model.gauss_params.means'].numpy()
    opacities = torch.sigmoid(
        state['pipeline']['_model.gauss_params.opacities']).numpy().flatten()
    print(f'\nGaussian 분포:')
    print(f'  N={len(means):,}')
    print(f'  X: [{means[:,0].min():.1f}, {means[:,0].max():.1f}]  range={np.ptp(means[:,0]):.1f}m')
    print(f'  Y: [{means[:,1].min():.1f}, {means[:,1].max():.1f}]  range={np.ptp(means[:,1]):.1f}m')
    print(f'  Z: [{means[:,2].min():.1f}, {means[:,2].max():.1f}]  range={np.ptp(means[:,2]):.1f}m')
    print(f'  opacity>0.5: {(opacities>0.5).sum():,} ({(opacities>0.5).mean()*100:.1f}%)')


--- AGS-Mesh 학습 ---  실험: indoor_scan_0528_045945
  진행: [....................]   2.2%  (   650/30000)
  경과: 00:06:39   ETA: 05:00:40
  Loss: -             PSNR: -
최근 로그:
  680 (2.27%)         120.857 ms           59 m, 3 s            22.97 M                                    
  690 (2.30%)         123.736 ms           1 h, 0 m, 26 s       22.44 M                                    
  700 (2.33%)         128.351 ms           1 h, 2 m, 40 s       21.60 M                                    
  710 (2.37%)         128.957 ms           1 h, 2 m, 57 s       21.52 M                                    
  720 (2.40%)         123.974 ms           1 h, 0 m, 29 s       22.40 M                                    
  730 (2.43%)         119.786 ms           58 m, 26 s           23.14 M                                    
  
  Step (% Done)       Train Iter (time)    ETA (time)           Train Rays / Sec     Test Rays / Sec       
  ----------------------------------------------------------------------

In [ ]:
#
# 학습 + Splat
#
import subprocess, os, re, sys, time, shutil
from pathlib import Path
from IPython.display import clear_output

os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']

BASE       = '/content/drive/MyDrive/3dgs_project'
PROC_DIR   = f'{BASE}/processed'
OUTPUT_DIR = f'{BASE}/output'
os.makedirs(f'{OUTPUT_DIR}/logs', exist_ok=True)

EXP_NAME  = f'splat_15k_{time.strftime("%m%d_%H%M%S")}'
MAX_ITER  = 15000
LOG_EVERY = 50
BACKUP_STEPS = {5000, 10000, 15000}

cmd = [
    '/usr/local/bin/ns-train',
    'dn-splatter',
    '--output-dir', OUTPUT_DIR,
    '--experiment-name', EXP_NAME,
    '--max-num-iterations', str(MAX_ITER),
    '--steps-per-save', '5000',
    '--vis', 'tensorboard',
    '--pipeline.model.use-depth-loss', 'True',
    '--pipeline.model.depth-lambda', '0.03',
    '--pipeline.model.use-normal-loss', 'True',
    '--pipeline.model.use-normal-tv-loss', 'False',
    '--pipeline.model.normal-supervision', 'depth',
    'normal-nerfstudio',
    '--data', PROC_DIR,
    '--load-3D-points', 'True',
    '--load-normals', 'False',
    '--load-depth-confidence-masks', 'False',
    '--downscale-factor', '1',
    '--scene-scale', '75.0',
    '--auto-scale-poses', 'False',
]

print(f'실험: {EXP_NAME}')
print(f'depth-lambda=0.03  normal-tv=False  15k iter')
print(f'백업 step: {BACKUP_STEPS}')
pat_step = re.compile(r'^\s*(\d+)\s*\(')
pat_loss = re.compile(r'loss[:\s]+([\d\.eE+\-]+)', re.I)
pat_psnr = re.compile(r'psnr[:\s]+([\d\.]+)', re.I)
last_step, last_loss, last_psnr = 0, '-', '-'
backed_up = set()
recent_lines = []
start_time = time.time()

def try_backup(step):
    if step not in BACKUP_STEPS or step in backed_up:
        return
    ckpt_dirs = sorted(
        Path(OUTPUT_DIR).rglob(f'{EXP_NAME}/dn-splatter/*/nerfstudio_models'),
        key=lambda x: x.stat().st_mtime)
    if not ckpt_dirs:
        return
    ckpt_dir = ckpt_dirs[-1]
    ckpts = sorted(ckpt_dir.glob('*.ckpt'))
    if not ckpts:
        return
    latest = ckpts[-1]
    backup_path = ckpt_dir / f'backup_step{step:06d}.ckpt'
    shutil.copy(str(latest), str(backup_path))
    backed_up.add(step)
    print(f'\n[백업] step={step:,} → {backup_path.name}')

def render_status():
    elapsed = time.time() - start_time
    pct = last_step / MAX_ITER * 100
    bar = '#' * int(pct/5) + '.' * (20 - int(pct/5))
    eta_str = time.strftime('%H:%M:%S', time.gmtime(
        elapsed / last_step * (MAX_ITER - last_step))) if last_step > 0 else '--:--:--'
    clear_output(wait=True)
    print(f'--- DN-Splatter ---  실험: {EXP_NAME}')
    print(f'  진행: [{bar}] {pct:5.1f}%  ({last_step:>6}/{MAX_ITER})')
    print(f'  경과: {time.strftime("%H:%M:%S", time.gmtime(elapsed))}   ETA: {eta_str}')
    print(f'  Loss: {last_loss:<12}  PSNR: {last_psnr}')
    print(f'  백업 완료: {sorted(backed_up)}')
    print('최근 로그:')
    for l in recent_lines[-10:]: print(' ', l)

# ── 학습 ──────────────────────────────────────────────────────────────
with open(f'{OUTPUT_DIR}/logs/training_log.txt', 'w') as logf:
    proc = subprocess.Popen(
        cmd, cwd='/content/dn-splatter',
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1)
    for raw_line in proc.stdout:
        for line in raw_line.replace('\r', '\n').split('\n'):
            line = line.strip()
            if not line: continue
            logf.write(line + '\n'); logf.flush()
            recent_lines.append(line)
            if len(recent_lines) > 100: recent_lines.pop(0)
            m = pat_step.search(line)
            if m:
                last_step = int(m.group(1))
                try_backup(last_step)
            m = pat_loss.search(line)
            if m: last_loss = m.group(1)
            m = pat_psnr.search(line)
            if m: last_psnr = m.group(1)
            if last_step > 0 and last_step % LOG_EVERY == 0:
                render_status()
    proc.wait()

last_step = MAX_ITER; render_status()
print(f'\n소요: {(time.time()-start_time)/60:.1f}분  Return code: {proc.returncode}')

if proc.returncode == 0:
    import torch, numpy as np
    ckpts = sorted([c for c in Path(OUTPUT_DIR).rglob('*.ckpt')
                    if EXP_NAME in str(c)], key=lambda x: x.stat().st_mtime)
    print(f'\n저장된 체크포인트: {len(ckpts)}개')
    for ckpt in ckpts:
        state = torch.load(str(ckpt), map_location='cpu', weights_only=False)
        means = state['pipeline']['_model.gauss_params.means'].numpy()
        opacities = torch.sigmoid(
            state['pipeline']['_model.gauss_params.opacities']).numpy().flatten()
        scales = state['pipeline']['_model.gauss_params.scales'].numpy()
        print(f'  {ckpt.name}  N={len(means):,}  '
              f'opacity>0.5={(opacities>0.5).mean()*100:.1f}%  '
              f'scale={np.exp(scales).mean():.4f}')

    # ── 학습 완료 후 config 자동 탐색 → splat export ──────────────────
    print('\n--- Splat Export ---')
    configs = sorted(
        Path(OUTPUT_DIR).rglob(f'{EXP_NAME}/dn-splatter/*/config.yml'),
        key=lambda x: x.stat().st_mtime)

    if not configs:
        print('❌ config.yml 못 찾음')
    else:
        CONFIG   = str(configs[-1])
        CKPT_DIR = str(configs[-1].parent)
        print(f'config: {CONFIG}')

        export_proc = subprocess.run([
            '/usr/local/bin/ns-export',
            'gaussian-splat',
            '--load-config', CONFIG,
            '--output-dir', CKPT_DIR
        ], capture_output=True, text=True)

        print(export_proc.stdout[-500:])
        if export_proc.returncode != 0:
            print(export_proc.stderr[-300:])

        for f in Path(CKPT_DIR).rglob('splat.ply'):
            print(f'✅ {f.stat().st_size/1e6:.1f} MB  {f}')

--- DN-Splatter ---  실험: splat_15k_0530_050031
  진행: [####################] 100.0%  ( 15000/15000)
  경과: 01:13:02   ETA: 00:00:00
  Loss: -             PSNR: -
  백업 완료: [5000, 10000]
최근 로그:
  │   Config File          │ /content/drive/MyDrive/3dgs_project/output/splat_15k_0530_050031/dn-splatter/2026-05-30_…   │
  │   Checkpoint Directory │ /content/drive/MyDrive/3dgs_project/output/splat_15k_0530_050031/dn-splatter/2026-05-30_…   │
  │                        ╵                                                                                             │
  ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
  Printing profiling stats, from longest to shortest duration in seconds
  VanillaPipeline.get_eval_loss_dict: 0.3674
  Trainer.train_iteration: 0.2745
  VanillaPipeline.get_train_loss_dict: 0.2487
  VanillaPipeline.get_eval_image_metrics_and_images: 0.0771
  Trainer.eval_iteration: 0.0010

소요: 73.0분  Return code: 0


In [ ]:
# Splat용 학습
import subprocess, os, re, sys, time, shutil
from pathlib import Path
from IPython.display import clear_output

os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']

BASE       = '/content/drive/MyDrive/3dgs_project'
PROC_DIR   = f'{BASE}/processed'
OUTPUT_DIR = f'{BASE}/output'
os.makedirs(f'{OUTPUT_DIR}/logs', exist_ok=True)

EXP_NAME  = f'splat_15k_{time.strftime("%m%d_%H%M%S")}'
MAX_ITER  = 15000
LOG_EVERY = 50
BACKUP_STEPS = {5000, 10000, 15000}  # 이 step에서 백업

cmd = [
    '/usr/local/bin/ns-train',
    'dn-splatter',
    '--output-dir', OUTPUT_DIR,
    '--experiment-name', EXP_NAME,
    '--max-num-iterations', str(MAX_ITER),
    '--steps-per-save', '5000',
    '--vis', 'tensorboard',
    '--pipeline.model.use-depth-loss', 'True',
    '--pipeline.model.depth-lambda', '0.03',
    '--pipeline.model.use-normal-loss', 'True',
    '--pipeline.model.use-normal-tv-loss', 'False',
    '--pipeline.model.normal-supervision', 'depth',
    'normal-nerfstudio',
    '--data', PROC_DIR,
    '--load-3D-points', 'True',
    '--load-normals', 'False',
    '--load-depth-confidence-masks', 'False',
    '--downscale-factor', '1',
    '--scene-scale', '75.0',
    '--auto-scale-poses', 'False',
]

print(f'실험: {EXP_NAME}')
print(f'depth-lambda=0.03  normal-tv=False  cull-alpha=기본값  15k iter')
print(f'백업 step: {BACKUP_STEPS}')
pat_step = re.compile(r'^\s*(\d+)\s*\(')
pat_loss = re.compile(r'loss[:\s]+([\d\.eE+\-]+)', re.I)
pat_psnr = re.compile(r'psnr[:\s]+([\d\.]+)', re.I)
last_step, last_loss, last_psnr = 0, '-', '-'
backed_up = set()
recent_lines = []
start_time = time.time()

def try_backup(step):
    if step not in BACKUP_STEPS or step in backed_up:
        return
    ckpt_dirs = sorted(
        Path(OUTPUT_DIR).rglob(f'{EXP_NAME}/dn-splatter/*/nerfstudio_models'),
        key=lambda x: x.stat().st_mtime)
    if not ckpt_dirs:
        return
    ckpt_dir = ckpt_dirs[-1]
    ckpts = sorted(ckpt_dir.glob('*.ckpt'))
    if not ckpts:
        return
    latest = ckpts[-1]
    backup_name = f'backup_step{step:06d}.ckpt'
    backup_path = ckpt_dir / backup_name
    shutil.copy(str(latest), str(backup_path))
    backed_up.add(step)
    print(f'\n[백업] step={step:,} → {backup_name}')

def render_status():
    elapsed = time.time() - start_time
    pct = last_step / MAX_ITER * 100
    bar = '#' * int(pct/5) + '.' * (20 - int(pct/5))
    eta_str = time.strftime('%H:%M:%S', time.gmtime(
        elapsed / last_step * (MAX_ITER - last_step))) if last_step > 0 else '--:--:--'
    clear_output(wait=True)
    print(f'--- DN-Splatter ---  실험: {EXP_NAME}')
    print(f'  진행: [{bar}] {pct:5.1f}%  ({last_step:>6}/{MAX_ITER})')
    print(f'  경과: {time.strftime("%H:%M:%S", time.gmtime(elapsed))}   ETA: {eta_str}')
    print(f'  Loss: {last_loss:<12}  PSNR: {last_psnr}')
    print(f'  백업 완료: {sorted(backed_up)}')
    print('최근 로그:')
    for l in recent_lines[-10:]: print(' ', l)

with open(f'{OUTPUT_DIR}/logs/training_log.txt', 'w') as logf:
    proc = subprocess.Popen(
        cmd, cwd='/content/dn-splatter',
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1)
    for raw_line in proc.stdout:
        for line in raw_line.replace('\r', '\n').split('\n'):
            line = line.strip()
            if not line: continue
            logf.write(line + '\n'); logf.flush()
            recent_lines.append(line)
            if len(recent_lines) > 100: recent_lines.pop(0)
            m = pat_step.search(line)
            if m:
                last_step = int(m.group(1))
                try_backup(last_step)
            m = pat_loss.search(line)
            if m: last_loss = m.group(1)
            m = pat_psnr.search(line)
            if m: last_psnr = m.group(1)
            if last_step > 0 and last_step % LOG_EVERY == 0:
                render_status()
    proc.wait()

last_step = MAX_ITER; render_status()
print(f'\n소요: {(time.time()-start_time)/60:.1f}분  Return code: {proc.returncode}')

if proc.returncode == 0:
    import torch, numpy as np
    ckpts = sorted([c for c in Path(OUTPUT_DIR).rglob('*.ckpt')
                    if EXP_NAME in str(c)], key=lambda x: x.stat().st_mtime)
    print(f'\n저장된 체크포인트: {len(ckpts)}개')
    for ckpt in ckpts:
        step_num = int(ckpt.stem.replace('backup_step','').replace('step-','').lstrip('0') or '0')
        state = torch.load(str(ckpt), map_location='cpu', weights_only=False)
        means = state['pipeline']['_model.gauss_params.means'].numpy()
        opacities = torch.sigmoid(
            state['pipeline']['_model.gauss_params.opacities']).numpy().flatten()
        scales = state['pipeline']['_model.gauss_params.scales'].numpy()
        print(f'  {ckpt.name}  N={len(means):,}  '
              f'opacity>0.5={( opacities>0.5).mean()*100:.1f}%  '
              f'scale={np.exp(scales).mean():.4f}')

--- DN-Splatter ---  실험: splat_15k_0529_155906
  진행: [####################] 100.0%  ( 15000/15000)
  경과: 01:05:12   ETA: 00:00:00
  Loss: -             PSNR: -
  백업 완료: [5000, 10000]
최근 로그:
  │   Config File          │ /content/drive/MyDrive/3dgs_project/output/splat_15k_0529_155906/dn-splatter/2026-05-29_…   │
  │   Checkpoint Directory │ /content/drive/MyDrive/3dgs_project/output/splat_15k_0529_155906/dn-splatter/2026-05-29_…   │
  │                        ╵                                                                                             │
  ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
  Printing profiling stats, from longest to shortest duration in seconds
  VanillaPipeline.get_eval_loss_dict: 0.3001
  Trainer.train_iteration: 0.2413
  VanillaPipeline.get_train_loss_dict: 0.2160
  VanillaPipeline.get_eval_image_metrics_and_images: 0.0626
  Trainer.eval_iteration: 0.0008

소요: 65.2분  Return code: 0


In [ ]:
from pathlib import Path

OUTPUT_DIR = '/content/drive/MyDrive/3dgs_project/output/splat_15k_0529_155906/dn-splatter'
for p in sorted(Path(OUTPUT_DIR).iterdir()):
    print(p)

/content/drive/MyDrive/3dgs_project/output/splat_15k_0529_155906/dn-splatter/2026-05-29_155921


In [ ]:
# 세이브본으로 시작
import subprocess, os, re, sys, time
from pathlib import Path
from IPython.display import clear_output

os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']

BASE       = '/content/drive/MyDrive/3dgs_project'
PROC_DIR   = f'{BASE}/processed'
OUTPUT_DIR = f'{BASE}/output'

# ▶ 이어서 할 실험명 입력
EXP_NAME  = 'splat_30k_0529_053129'  # 끊긴 실험명으로 교체
MAX_ITER  = 30000
LOG_EVERY = 50

# 마지막 체크포인트 자동 탐색
exp_dirs = sorted(Path(OUTPUT_DIR).rglob(f'{EXP_NAME}/dn-splatter/*/nerfstudio_models'),
                  key=lambda x: x.stat().st_mtime)

if not exp_dirs:
    print('체크포인트 없음 - EXP_NAME 확인 필요')
else:
    CKPT_DIR = str(exp_dirs[-1])
    ckpts = sorted(Path(CKPT_DIR).glob('*.ckpt'))
    last_step = int(ckpts[-1].stem.split('-')[-1]) if ckpts else 0
    print(f'마지막 체크포인트: step={last_step:,}')
    print(f'이어서 학습: {last_step+1} → {MAX_ITER}')

    cmd = [
        '/usr/local/bin/ns-train',
        'dn-splatter',
        '--output-dir', OUTPUT_DIR,
        '--experiment-name', EXP_NAME,
        '--max-num-iterations', str(MAX_ITER),
        '--steps-per-save', '5000',
        '--load-dir', CKPT_DIR,      # ← 이어서 학습 핵심
        '--vis', 'tensorboard',
        '--pipeline.model.use-depth-loss', 'True',
        '--pipeline.model.depth-lambda', '0.03',
        '--pipeline.model.use-normal-loss', 'True',
        '--pipeline.model.use-normal-tv-loss', 'False',
        '--pipeline.model.normal-supervision', 'depth',
        '--pipeline.model.cull-alpha-thresh', '0.005',
        'normal-nerfstudio',
        '--data', PROC_DIR,
        '--load-3D-points', 'True',
        '--load-normals', 'False',
        '--load-depth-confidence-masks', 'False',
        '--downscale-factor', '1',
        '--scene-scale', '75.0',
        '--auto-scale-poses', 'False',
    ]

    pat_step = re.compile(r'^\s*(\d+)\s*\(')
    pat_loss = re.compile(r'loss[:\s]+([\d\.eE+\-]+)', re.I)
    pat_psnr = re.compile(r'psnr[:\s]+([\d\.]+)', re.I)
    last_step_log, last_loss, last_psnr = last_step, '-', '-'
    recent_lines = []
    start_time = time.time()

    def render_status():
        elapsed = time.time() - start_time
        pct = last_step_log / MAX_ITER * 100
        bar = '#' * int(pct/5) + '.' * (20 - int(pct/5))
        eta_str = time.strftime('%H:%M:%S', time.gmtime(
            elapsed / max(last_step_log - last_step, 1) *
            (MAX_ITER - last_step_log))) if last_step_log > last_step else '--:--:--'
        clear_output(wait=True)
        print(f'--- DN-Splatter (이어서) ---  실험: {EXP_NAME}')
        print(f'  진행: [{bar}] {pct:5.1f}%  ({last_step_log:>6}/{MAX_ITER})')
        print(f'  경과: {time.strftime("%H:%M:%S", time.gmtime(elapsed))}   ETA: {eta_str}')
        print(f'  Loss: {last_loss:<12}  PSNR: {last_psnr}')
        for l in recent_lines[-10:]: print(' ', l)

    with open(f'{OUTPUT_DIR}/logs/training_log_resume.txt', 'w') as logf:
        proc = subprocess.Popen(
            cmd, cwd='/content/dn-splatter',
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1)
        for raw_line in proc.stdout:
            for line in raw_line.replace('\r', '\n').split('\n'):
                line = line.strip()
                if not line: continue
                logf.write(line + '\n'); logf.flush()
                recent_lines.append(line)
                if len(recent_lines) > 100: recent_lines.pop(0)
                m = pat_step.search(line)
                if m: last_step_log = int(m.group(1))
                m = pat_loss.search(line)
                if m: last_loss = m.group(1)
                m = pat_psnr.search(line)
                if m: last_psnr = m.group(1)
                if last_step_log > 0 and last_step_log % LOG_EVERY == 0:
                    render_status()
        proc.wait()

    last_step_log = MAX_ITER; render_status()
    print(f'\n소요: {(time.time()-start_time)/60:.1f}분  Return code: {proc.returncode}')

--- DN-Splatter (이어서) ---  실험: splat_30k_0529_053129
  진행: [####################] 100.5%  ( 30150/30000)
  경과: 02:49:47   ETA: 23:58:59
  Loss: -             PSNR: -
  30080 (100.27%)     574.818 ms           23 h, 59 m, 15 s     4.92 M                                     
  30090 (100.30%)     580.305 ms           23 h, 59 m, 8 s      4.88 M                                     
  30100 (100.33%)     601.190 ms           23 h, 59 m, 0 s      4.75 M                                     
  30110 (100.37%)     600.117 ms           23 h, 58 m, 54 s     4.75 M                                     
  30120 (100.40%)     580.488 ms           23 h, 58 m, 51 s     4.87 M                                     
  30130 (100.43%)     579.354 ms           23 h, 58 m, 45 s     4.88 M                                     
  30140 (100.47%)     577.039 ms           23 h, 58 m, 40 s     4.90 M                                     
  30150 (100.50%)     576.581 ms           23 h, 58 m, 34 s     4.90 M        

KeyboardInterrupt: 

In [ ]:
import torch, numpy as np
from pathlib import Path

BASE = '/content/drive/MyDrive/3dgs_project'

ckpts = {
    '5k':  f'{BASE}/output/splat_30k_0529_053129/dn-splatter/2026-05-29_053144/nerfstudio_models/step-000005000.ckpt',
    '30k': f'{BASE}/output/splat_30k_0529_053129/dn-splatter/2026-05-29_072713/nerfstudio_models/step-000030000.ckpt',
}

for label, ckpt in ckpts.items():
    state = torch.load(ckpt, map_location='cpu', weights_only=False)
    means = state['pipeline']['_model.gauss_params.means'].numpy()
    opacities = torch.sigmoid(
        state['pipeline']['_model.gauss_params.opacities']).numpy().flatten()
    scales = state['pipeline']['_model.gauss_params.scales'].numpy()
    print(f'[{label}]')
    print(f'  N={len(means):,}')
    print(f'  opacity>0.5: {(opacities>0.5).mean()*100:.1f}%')
    print(f'  opacity>0.1: {(opacities>0.1).mean()*100:.1f}%')
    print(f'  scale mean: {np.exp(scales).mean():.4f}')

[5k]
  N=64,361
  opacity>0.5: 34.1%
  opacity>0.1: 55.7%
  scale mean: 0.0450
[30k]
  N=17,559
  opacity>0.5: 76.0%
  opacity>0.1: 83.3%
  scale mean: 0.0715


In [ ]:
from pathlib import Path

BASE = '/content/drive/MyDrive/3dgs_project'

# 최근 수정된 모든 ckpt 확인
ckpts = sorted(Path(f'{BASE}/output').rglob('*.ckpt'),
               key=lambda x: x.stat().st_mtime, reverse=True)
for c in ckpts[:10]:
    print(f'{c.parent.parent.parent.name}  step={int(c.stem.split("-")[-1]):,}  '
          f'{c.stat().st_size/1e6:.1f}MB  {c}')

dn-splatter  step=30,000  32.4MB  /content/drive/MyDrive/3dgs_project/output/splat_30k_0529_053129/dn-splatter/2026-05-29_072713/nerfstudio_models/step-000030000.ckpt
dn-splatter  step=5,000  66.1MB  /content/drive/MyDrive/3dgs_project/output/splat_30k_0529_053129/dn-splatter/2026-05-29_053144/nerfstudio_models/step-000005000.ckpt
dn-splatter  step=9,999  107.8MB  /content/drive/MyDrive/3dgs_project/output/splat_quality_0528_164809/dn-splatter/2026-05-28_164838/nerfstudio_models/step-000009999.ckpt
ags-mesh  step=9,999  68.1MB  /content/drive/MyDrive/3dgs_project/output/indoor_scan_depthsafe_0528_112051/ags-mesh/2026-05-28_112107/nerfstudio_models/step-000009999.ckpt
ags-mesh  step=9,999  68.3MB  /content/drive/MyDrive/3dgs_project/output/indoor_scan_depthsafe_0528_102411/ags-mesh/2026-05-28_102428/nerfstudio_models/step-000009999.ckpt
ags-mesh  step=6,999  70.0MB  /content/drive/MyDrive/3dgs_project/output/indoor_scan_depthsafe_0528_094326/ags-mesh/2026-05-28_094343/nerfstudio_models/

In [ ]:
import subprocess, os
from pathlib import Path

os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']

BASE     = '/content/drive/MyDrive/3dgs_project'
CONFIG   = f'{BASE}/output/splat_15k_0529_155906/dn-splatter/2026-05-29_155921/config.yml'
CKPT_DIR = f'{BASE}/output/splat_15k_0529_155906/dn-splatter/2026-05-29_155921'

print(f'config: {CONFIG}')

proc = subprocess.run([
    '/usr/local/bin/ns-export',
    'gaussian-splat',
    '--load-config', CONFIG,
    '--output-dir', CKPT_DIR
], capture_output=True, text=True)

print(proc.stdout[-500:])
if proc.returncode != 0:
    print(proc.stderr[-300:])

for f in Path(CKPT_DIR).rglob('splat.ply'):
    print(f'✅ {f.stat().st_size/1e6:.1f} MB  {f}')

config: /content/drive/MyDrive/3dgs_project/output/splat_15k_0529_155906/dn-splatter/2026-05-29_155921/config.yml
[17:10:18] Number of initial seed points 96000                                                           dn_model.py:136
Initialising Gaussian normals from intial seed points
Loading latest checkpoint from load_dir
✅ Done loading checkpoint from 
/content/drive/MyDrive/3dgs_project/output/splat_15k_0529_155906/dn-splatter/2026-05-29_155921/nerfstudio_models/step-00
0014999.ckpt

✅ 258.2 MB  /content/drive/MyDrive/3dgs_project/output/splat_15k_0529_155906/dn-splatter/2026-05-29_155921/splat.ply


In [ ]:
from pathlib import Path

OUTPUT_DIR = '/content/drive/MyDrive/3dgs_project/output/splat_15k_0529_135901/dn-splatter'
for p in sorted(Path(OUTPUT_DIR).iterdir()):
    print(p)

/content/drive/MyDrive/3dgs_project/output/splat_15k_0529_135901/dn-splatter/2026-05-29_135901
/content/drive/MyDrive/3dgs_project/output/splat_15k_0529_135901/dn-splatter/2026-05-29_135927


In [ ]:
import torch, numpy as np
from pathlib import Path

BASE = '/content/drive/MyDrive/3dgs_project'

ckpts = sorted(Path(f'{BASE}/output').rglob('step-000029999.ckpt'),
               key=lambda x: x.stat().st_mtime)

for ckpt in ckpts:
    try:
        state = torch.load(str(ckpt), map_location='cpu', weights_only=False)
        means = state['pipeline']['_model.gauss_params.means'].numpy()
        opacities = torch.sigmoid(
            state['pipeline']['_model.gauss_params.opacities']).numpy().flatten()
        exp = ckpt.parent.parent.parent.name
        method = ckpt.parent.parent.name
        print(f'{exp}/{method}')
        print(f'  N={len(means):,}  opacity>0.5={( opacities>0.5).mean()*100:.1f}%  Z_range={np.ptp(means[:,2]):.1f}m')
    except Exception as e:
        print(f'  에러: {e}')

dn-splatter/2026-03-22_170130
  N=605,498  opacity>0.5=87.6%  Z_range=10.0m
dn-splatter/2026-04-14_150311
  N=653,530  opacity>0.5=59.1%  Z_range=10.0m
dn-splatter/2026-05-12_150126
  N=269,630  opacity>0.5=57.4%  Z_range=10.3m
ags-mesh/2026-05-14_124321
  N=441,197  opacity>0.5=2.1%  Z_range=15.6m
ags-mesh/2026-05-18_122624
  N=253,999  opacity>0.5=92.7%  Z_range=7.8m
ags-mesh/2026-05-22_063037
  N=273,697  opacity>0.5=90.4%  Z_range=8.3m
ags-mesh/2026-05-26_135646
  N=13,861  opacity>0.5=95.8%  Z_range=6.1m
ags-mesh/2026-05-26_173436
  N=13,748  opacity>0.5=95.7%  Z_range=7.4m
ags-mesh/2026-05-27_035954
  N=102,325  opacity>0.5=68.8%  Z_range=9.0m
ags-mesh/2026-05-28_050002
  N=56,773  opacity>0.5=97.0%  Z_range=6.8m


In [ ]:
import subprocess, sys, os
from pathlib import Path

os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']

BASE = '/content/drive/MyDrive/3dgs_project'

targets = [
    ('ags-mesh', '2026-05-14_124321'),
    ('ags-mesh', '2026-05-26_135646'),
    ('ags-mesh', '2026-05-26_173436'),
    ('ags-mesh', '2026-05-27_035954'),
    ('ags-mesh', '2026-05-28_050002'),
]

for method, date in targets:
    configs = list(Path(f'{BASE}/output').rglob(f'{date}/config.yml'))
    if not configs:
        print(f'[{method}/{date}] config.yml 없음 - 스킵')
        continue

    CONFIG   = str(configs[0])
    CKPT_DIR = str(configs[0].parent)
    out_ply  = Path(CKPT_DIR) / 'splat.ply'

    if out_ply.exists():
        print(f'[{method}/{date}] 이미 존재: {out_ply.stat().st_size/1e6:.1f} MB')
        continue

    print(f'[{method}/{date}] export 중...')
    proc = subprocess.run([
        sys.executable, '-m', 'nerfstudio.scripts.exporter',
        'gaussian-splat',
        '--load-config', CONFIG,
        '--output-dir', CKPT_DIR
    ], capture_output=True, text=True)

    if proc.returncode == 0:
        for f in Path(CKPT_DIR).rglob('splat.ply'):
            print(f'  ✅ {f.stat().st_size/1e6:.1f} MB  {f}')
    else:
        print(f'  ❌ 실패: {proc.stderr[-200:]}')

[ags-mesh/2026-05-14_124321] export 중...
  ✅ 109.4 MB  /content/drive/MyDrive/3dgs_project/output/indoor_scan/ags-mesh/2026-05-14_124321/splat.ply
[ags-mesh/2026-05-26_135646] export 중...
  ✅ 3.4 MB  /content/drive/MyDrive/3dgs_project/output/indoor_scan_0526_135629/ags-mesh/2026-05-26_135646/splat.ply
[ags-mesh/2026-05-26_173436] export 중...
  ✅ 3.4 MB  /content/drive/MyDrive/3dgs_project/output/indoor_scan_0526_173420/ags-mesh/2026-05-26_173436/splat.ply
[ags-mesh/2026-05-27_035954] export 중...
  ✅ 25.4 MB  /content/drive/MyDrive/3dgs_project/output/indoor_scan_0527_035938/ags-mesh/2026-05-27_035954/splat.ply
[ags-mesh/2026-05-28_050002] export 중...
  ✅ 14.1 MB  /content/drive/MyDrive/3dgs_project/output/indoor_scan_0528_045945/ags-mesh/2026-05-28_050002/splat.ply


In [ ]:
import subprocess, os, re, sys, time
from pathlib import Path
from IPython.display import clear_output

os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']

BASE       = '/content/drive/MyDrive/3dgs_project'
PROC_DIR   = f'{BASE}/processed'
OUTPUT_DIR = f'{BASE}/output'
os.makedirs(f'{OUTPUT_DIR}/logs', exist_ok=True)

EXP_NAME  = f'splat_quality_{time.strftime("%m%d_%H%M%S")}'
MAX_ITER  = 10000
LOG_EVERY = 100

cmd = [
    sys.executable, '-m', 'nerfstudio.scripts.train',
    'dn-splatter',
    '--output-dir', OUTPUT_DIR,
    '--experiment-name', EXP_NAME,
    '--max-num-iterations', str(MAX_ITER),
    '--vis', 'tensorboard',
    '--pipeline.model.use-depth-loss', 'True',
    '--pipeline.model.depth-lambda', '0.03',
    '--pipeline.model.use-normal-loss', 'True',
    '--pipeline.model.use-normal-tv-loss', 'False',
    '--pipeline.model.normal-supervision', 'depth',
    '--pipeline.model.cull-alpha-thresh', '0.005',
    'normal-nerfstudio',
    '--data', PROC_DIR,
    '--load-3D-points', 'True',
    '--load-normals', 'False',
    '--load-depth-confidence-masks', 'False',
    '--downscale-factor', '1',
    '--scene-scale', '75.0',
    '--auto-scale-poses', 'False',
]

print(f'실험: {EXP_NAME}')
print(f'depth-lambda=0.03  normal-tv=False  cull-alpha=0.005  10k iter')
pat_step = re.compile(r'^\s*(\d+)\s*\(')
pat_loss = re.compile(r'loss[:\s]+([\d\.eE+\-]+)', re.I)
pat_psnr = re.compile(r'psnr[:\s]+([\d\.]+)', re.I)
last_step, last_loss, last_psnr = 0, '-', '-'
recent_lines = []
start_time = time.time()

def render_status():
    elapsed = time.time() - start_time
    pct = last_step / MAX_ITER * 100
    bar = '#' * int(pct/5) + '.' * (20 - int(pct/5))
    eta_str = time.strftime('%H:%M:%S', time.gmtime(
        elapsed / last_step * (MAX_ITER - last_step))) if last_step > 0 else '--:--:--'
    clear_output(wait=True)
    print(f'--- DN-Splatter (splat 품질) ---  실험: {EXP_NAME}')
    print(f'  진행: [{bar}] {pct:5.1f}%  ({last_step:>6}/{MAX_ITER})')
    print(f'  경과: {time.strftime("%H:%M:%S", time.gmtime(elapsed))}   ETA: {eta_str}')
    print(f'  Loss: {last_loss:<12}  PSNR: {last_psnr}')
    print('최근 로그:')
    for l in recent_lines[-10:]: print(' ', l)

with open(f'{OUTPUT_DIR}/logs/training_log.txt', 'w') as logf:
    proc = subprocess.Popen(
        cmd, cwd='/content/dn-splatter',
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1)
    for raw_line in proc.stdout:
        for line in raw_line.replace('\r', '\n').split('\n'):
            line = line.strip()
            if not line: continue
            logf.write(line + '\n'); logf.flush()
            recent_lines.append(line)
            if len(recent_lines) > 100: recent_lines.pop(0)
            m = pat_step.search(line)
            if m: last_step = int(m.group(1))
            m = pat_loss.search(line)
            if m: last_loss = m.group(1)
            m = pat_psnr.search(line)
            if m: last_psnr = m.group(1)
            if last_step > 0 and last_step % LOG_EVERY == 0:
                render_status()
    proc.wait()

last_step = MAX_ITER; render_status()
print(f'\n소요: {(time.time()-start_time)/60:.1f}분  Return code: {proc.returncode}')

if proc.returncode == 0:
    import torch, numpy as np
    ckpts = sorted([c for c in Path(OUTPUT_DIR).rglob('*.ckpt')
                    if EXP_NAME in str(c)], key=lambda x: x.stat().st_mtime)
    state = torch.load(str(ckpts[-1]), map_location='cpu', weights_only=False)
    means = state['pipeline']['_model.gauss_params.means'].numpy()
    opacities = torch.sigmoid(
        state['pipeline']['_model.gauss_params.opacities']).numpy().flatten()
    print(f'\nGaussian 분포:')
    print(f'  N={len(means):,}')
    print(f'  opacity>0.5: {(opacities>0.5).sum():,} ({(opacities>0.5).mean()*100:.1f}%)')

--- DN-Splatter (splat 품질) ---  실험: splat_quality_0528_164809
  진행: [####################] 100.0%  ( 10000/10000)
  경과: 01:03:20   ETA: 00:00:00
  Loss: -             PSNR: -
최근 로그:
  │   Config File          │ /content/drive/MyDrive/3dgs_project/output/splat_quality_0528_164809/dn-splatter/2026-05…   │
  │   Checkpoint Directory │ /content/drive/MyDrive/3dgs_project/output/splat_quality_0528_164809/dn-splatter/2026-05…   │
  │                        ╵                                                                                             │
  ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
  Printing profiling stats, from longest to shortest duration in seconds
  VanillaPipeline.get_eval_loss_dict: 7.7744
  Trainer.train_iteration: 0.3452
  VanillaPipeline.get_train_loss_dict: 0.3206
  VanillaPipeline.get_eval_image_metrics_and_images: 0.0730
  Trainer.eval_iteration: 0.0150

소요: 63.3분  Return code: 0

Gaussia

In [ ]:
import subprocess, sys, os
from pathlib import Path

os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']

BASE       = '/content/drive/MyDrive/3dgs_project'
OUTPUT_DIR = f'{BASE}/output'

configs = sorted(
    [c for c in Path(OUTPUT_DIR).rglob('config.yml')
     if 'splat_quality_0528_164809' in str(c)
     and (c.parent / 'nerfstudio_models').exists()],
    key=lambda x: x.stat().st_mtime
)
CONFIG   = str(configs[-1])
CKPT_DIR = str(configs[-1].parent)
print(f'config: {CONFIG}')

proc = subprocess.run([
    sys.executable, '-m', 'nerfstudio.scripts.exporter',
    'gaussian-splat',
    '--load-config', CONFIG,
    '--output-dir', CKPT_DIR
], capture_output=True, text=True)

if proc.returncode == 0:
    for f in Path(CKPT_DIR).rglob('splat.ply'):
        print(f'✅ {f.stat().st_size/1e6:.1f} MB  {f}')
else:
    print(f'❌ {proc.stderr[-300:]}')

config: /content/drive/MyDrive/3dgs_project/output/splat_quality_0528_164809/dn-splatter/2026-05-28_164838/config.yml
✅ 30.3 MB  /content/drive/MyDrive/3dgs_project/output/splat_quality_0528_164809/dn-splatter/2026-05-28_164838/splat.ply


In [ ]:
import subprocess, sys, os
from pathlib import Path

os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']

BASE = '/content/drive/MyDrive/3dgs_project'

targets = [
    ('ags-mesh', '2026-05-26_135646'),
    ('ags-mesh', '2026-05-26_173436'),
    ('ags-mesh', '2026-05-28_050002'),
]

for method, date in targets:
    configs = list(Path(f'{BASE}/output').rglob(f'{date}/config.yml'))
    if not configs:
        print(f'[{date}] config.yml 없음 - 스킵')
        continue

    CONFIG   = str(configs[0])
    CKPT_DIR = str(configs[0].parent)
    out_ply  = Path(CKPT_DIR) / 'splat.ply'

    if out_ply.exists():
        print(f'[{date}] 이미 존재: {out_ply.stat().st_size/1e6:.1f} MB')
        continue

    print(f'[{date}] export 중...')
    proc = subprocess.run([
        sys.executable, '-m', 'nerfstudio.scripts.exporter',
        'gaussian-splat',
        '--load-config', CONFIG,
        '--output-dir', CKPT_DIR
    ], capture_output=True, text=True)

    if proc.returncode == 0:
        for f in Path(CKPT_DIR).rglob('splat.ply'):
            print(f'  ✅ {f.stat().st_size/1e6:.1f} MB  {f}')
    else:
        print(f'  ❌ {proc.stderr[-200:]}')

[2026-05-26_135646] 이미 존재: 3.4 MB
[2026-05-26_173436] 이미 존재: 3.4 MB
[2026-05-28_050002] 이미 존재: 14.1 MB


In [ ]:
from pathlib import Path

BASE = '/content/drive/MyDrive/3dgs_project'

targets = [
    '2026-05-26_135646',
    '2026-05-26_173436',
    '2026-05-28_050002',
]

for date in targets:
    configs = list(Path(f'{BASE}/output').rglob(f'{date}/config.yml'))
    if configs:
        print(f'[{date}] {configs[0].parent.parent.parent.name}')

[2026-05-26_135646] indoor_scan_0526_135629
[2026-05-26_173436] indoor_scan_0526_173420
[2026-05-28_050002] indoor_scan_0528_045945


In [ ]:
import subprocess, os, re, sys, time
from pathlib import Path
from IPython.display import clear_output

os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']

BASE       = '/content/drive/MyDrive/3dgs_project'
PROC_DIR   = f'{BASE}/processed'
OUTPUT_DIR = f'{BASE}/output'
os.makedirs(f'{OUTPUT_DIR}/logs', exist_ok=True)

EXP_NAME  = f'indoor_scan_{time.strftime("%m%d_%H%M%S")}'
MAX_ITER  = 30000
LOG_EVERY = 50

cmd = [
    sys.executable, '-m', 'nerfstudio.scripts.train',
    'ags-mesh',
    '--output-dir', OUTPUT_DIR,
    '--experiment-name', EXP_NAME,
    '--max-num-iterations', str(MAX_ITER),
    '--vis', 'tensorboard',
    '--pipeline.model.use-depth-loss', 'True',
    '--pipeline.model.depth-lambda', '0.2',
    '--pipeline.model.use-normal-loss', 'True',
    '--pipeline.model.use-normal-tv-loss', 'True',
    '--pipeline.model.normal-supervision', 'depth',
    'normal-nerfstudio',
    '--data', PROC_DIR,
    '--load-3D-points', 'True',
    '--load-normals', 'False',
    '--load-depth-confidence-masks', 'False',
    '--downscale-factor', '1',
    '--scene-scale', '110.0',       # 71m × 1.5 = 107 → 여유있게 110
    '--auto-scale-poses', 'False',
]

print(f'실험: {EXP_NAME}')
pat_step = re.compile(r'^\s*(\d+)\s*\(')
pat_loss = re.compile(r'loss[:\s]+([\d\.eE+\-]+)', re.I)
pat_psnr = re.compile(r'psnr[:\s]+([\d\.]+)', re.I)
last_step, last_loss, last_psnr = 0, '-', '-'
recent_lines = []
start_time = time.time()

def render_status():
    elapsed = time.time() - start_time
    pct = last_step / MAX_ITER * 100
    bar = '#' * int(pct/5) + '.' * (20 - int(pct/5))
    eta_str = time.strftime('%H:%M:%S', time.gmtime(
        elapsed / last_step * (MAX_ITER - last_step))) if last_step > 0 else '--:--:--'
    clear_output(wait=True)
    print(f'--- AGS-Mesh 학습 ---  실험: {EXP_NAME}')
    print(f'  진행: [{bar}] {pct:5.1f}%  ({last_step:>6}/{MAX_ITER})')
    print(f'  경과: {time.strftime("%H:%M:%S", time.gmtime(elapsed))}   ETA: {eta_str}')
    print(f'  Loss: {last_loss:<12}  PSNR: {last_psnr}')
    print('최근 로그:')
    for l in recent_lines[-10:]: print(' ', l)

with open(f'{OUTPUT_DIR}/logs/training_log.txt', 'w') as logf:
    proc = subprocess.Popen(
        cmd, cwd='/content/dn-splatter',
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1)
    for raw_line in proc.stdout:
        for line in raw_line.replace('\r', '\n').split('\n'):
            line = line.strip()
            if not line: continue
            logf.write(line + '\n'); logf.flush()
            recent_lines.append(line)
            if len(recent_lines) > 100: recent_lines.pop(0)
            m = pat_step.search(line)
            if m: last_step = int(m.group(1))
            m = pat_loss.search(line)
            if m: last_loss = m.group(1)
            m = pat_psnr.search(line)
            if m: last_psnr = m.group(1)
            if last_step > 0 and last_step % LOG_EVERY == 0:
                render_status()
    proc.wait()

last_step = MAX_ITER; render_status()
print(f'\n소요: {(time.time()-start_time)/60:.1f}분  Return code: {proc.returncode}')

if proc.returncode == 0:
    import torch, numpy as np
    ckpts = sorted([c for c in Path(OUTPUT_DIR).rglob('*.ckpt')
                    if EXP_NAME in str(c)], key=lambda x: x.stat().st_mtime)
    state = torch.load(str(ckpts[-1]), map_location='cpu', weights_only=False)
    means = state['pipeline']['_model.gauss_params.means'].numpy()
    opacities = torch.sigmoid(
        state['pipeline']['_model.gauss_params.opacities']).numpy().flatten()
    print(f'\nGaussian 분포:')
    print(f'  N={len(means):,}')
    print(f'  X: [{means[:,0].min():.1f}, {means[:,0].max():.1f}]  range={np.ptp(means[:,0]):.1f}m')
    print(f'  Y: [{means[:,1].min():.1f}, {means[:,1].max():.1f}]  range={np.ptp(means[:,1]):.1f}m')
    print(f'  Z: [{means[:,2].min():.1f}, {means[:,2].max():.1f}]  range={np.ptp(means[:,2]):.1f}m')
    print(f'  opacity>0.5: {(opacities>0.5).sum():,} ({(opacities>0.5).mean()*100:.1f}%)')

--- AGS-Mesh 학습 ---  실험: indoor_scan_0526_173420
  진행: [####################] 100.0%  ( 30000/30000)
  경과: 03:23:10   ETA: 00:00:00
  Loss: -             PSNR: -
최근 로그:
  │   Config File          │ /content/drive/MyDrive/3dgs_project/output/indoor_scan_0526_173420/ags-mesh/2026-05-26_1…   │
  │   Checkpoint Directory │ /content/drive/MyDrive/3dgs_project/output/indoor_scan_0526_173420/ags-mesh/2026-05-26_1…   │
  │                        ╵                                                                                             │
  ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
  Printing profiling stats, from longest to shortest duration in seconds
  VanillaPipeline.get_eval_loss_dict: 0.5224
  Trainer.train_iteration: 0.3895
  VanillaPipeline.get_train_loss_dict: 0.3648
  VanillaPipeline.get_eval_image_metrics_and_images: 0.0703
  Trainer.eval_iteration: 0.0013

소요: 203.2분  Return code: 0

Gaussian 분포:
  N=13

In [ ]:
import subprocess, os, re, sys, time
from pathlib import Path
from IPython.display import clear_output

os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']

BASE       = '/content/drive/MyDrive/3dgs_project'
PROC_DIR   = f'{BASE}/processed'
OUTPUT_DIR = f'{BASE}/output'
os.makedirs(f'{OUTPUT_DIR}/logs', exist_ok=True)

EXP_NAME  = f'indoor_scan_{time.strftime("%m%d_%H%M%S")}'
MAX_ITER  = 30000
LOG_EVERY = 50

cmd = [
    sys.executable, '-m', 'nerfstudio.scripts.train',
    'ags-mesh',
    '--output-dir', OUTPUT_DIR,
    '--experiment-name', EXP_NAME,
    '--max-num-iterations', str(MAX_ITER),
    '--vis', 'tensorboard',
    '--pipeline.model.use-depth-loss', 'True',
    '--pipeline.model.depth-lambda', '0.2',
    '--pipeline.model.use-normal-loss', 'True',
    '--pipeline.model.use-normal-tv-loss', 'True',
    '--pipeline.model.normal-supervision', 'depth',
    '--pipeline.model.cull-alpha-thresh', '0.005',   # ← 기본 0.1에서 낮춤
    '--pipeline.model.densify-grad-thresh', '0.0001', # ← densification 민감도
    'normal-nerfstudio',
    '--data', PROC_DIR,
    '--load-3D-points', 'True',
    '--load-normals', 'False',
    '--load-depth-confidence-masks', 'False',
    '--downscale-factor', '1',
    '--scene-scale', '110.0',
    '--auto-scale-poses', 'False',
]

print(f'실험: {EXP_NAME}')
pat_step = re.compile(r'^\s*(\d+)\s*\(')
pat_loss = re.compile(r'loss[:\s]+([\d\.eE+\-]+)', re.I)
pat_psnr = re.compile(r'psnr[:\s]+([\d\.]+)', re.I)
last_step, last_loss, last_psnr = 0, '-', '-'
recent_lines = []
start_time = time.time()

def render_status():
    elapsed = time.time() - start_time
    pct = last_step / MAX_ITER * 100
    bar = '#' * int(pct/5) + '.' * (20 - int(pct/5))
    eta_str = time.strftime('%H:%M:%S', time.gmtime(
        elapsed / last_step * (MAX_ITER - last_step))) if last_step > 0 else '--:--:--'
    clear_output(wait=True)
    print(f'--- AGS-Mesh 학습 ---  실험: {EXP_NAME}')
    print(f'  진행: [{bar}] {pct:5.1f}%  ({last_step:>6}/{MAX_ITER})')
    print(f'  경과: {time.strftime("%H:%M:%S", time.gmtime(elapsed))}   ETA: {eta_str}')
    print(f'  Loss: {last_loss:<12}  PSNR: {last_psnr}')
    print('최근 로그:')
    for l in recent_lines[-10:]: print(' ', l)

with open(f'{OUTPUT_DIR}/logs/training_log.txt', 'w') as logf:
    proc = subprocess.Popen(
        cmd, cwd='/content/dn-splatter',
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1)
    for raw_line in proc.stdout:
        for line in raw_line.replace('\r', '\n').split('\n'):
            line = line.strip()
            if not line: continue
            logf.write(line + '\n'); logf.flush()
            recent_lines.append(line)
            if len(recent_lines) > 100: recent_lines.pop(0)
            m = pat_step.search(line)
            if m: last_step = int(m.group(1))
            m = pat_loss.search(line)
            if m: last_loss = m.group(1)
            m = pat_psnr.search(line)
            if m: last_psnr = m.group(1)
            if last_step > 0 and last_step % LOG_EVERY == 0:
                render_status()
    proc.wait()

last_step = MAX_ITER; render_status()
print(f'\n소요: {(time.time()-start_time)/60:.1f}분  Return code: {proc.returncode}')

if proc.returncode == 0:
    import torch, numpy as np
    ckpts = sorted([c for c in Path(OUTPUT_DIR).rglob('*.ckpt')
                    if EXP_NAME in str(c)], key=lambda x: x.stat().st_mtime)
    state = torch.load(str(ckpts[-1]), map_location='cpu', weights_only=False)
    means = state['pipeline']['_model.gauss_params.means'].numpy()
    opacities = torch.sigmoid(
        state['pipeline']['_model.gauss_params.opacities']).numpy().flatten()
    print(f'\nGaussian 분포:')
    print(f'  N={len(means):,}')
    print(f'  X: [{means[:,0].min():.1f}, {means[:,0].max():.1f}]  range={np.ptp(means[:,0]):.1f}m')
    print(f'  Y: [{means[:,1].min():.1f}, {means[:,1].max():.1f}]  range={np.ptp(means[:,1]):.1f}m')
    print(f'  Z: [{means[:,2].min():.1f}, {means[:,2].max():.1f}]  range={np.ptp(means[:,2]):.1f}m')
    print(f'  opacity>0.5: {(opacities>0.5).sum():,} ({(opacities>0.5).mean()*100:.1f}%)')

--- AGS-Mesh 학습 ---  실험: indoor_scan_0527_035938
  진행: [####################] 100.0%  ( 30000/30000)
  경과: 03:33:10   ETA: 00:00:00
  Loss: -             PSNR: -
최근 로그:
  │   Config File          │ /content/drive/MyDrive/3dgs_project/output/indoor_scan_0527_035938/ags-mesh/2026-05-27_0…   │
  │   Checkpoint Directory │ /content/drive/MyDrive/3dgs_project/output/indoor_scan_0527_035938/ags-mesh/2026-05-27_0…   │
  │                        ╵                                                                                             │
  ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
  Printing profiling stats, from longest to shortest duration in seconds
  VanillaPipeline.get_eval_loss_dict: 2.6831
  Trainer.train_iteration: 0.4052
  VanillaPipeline.get_train_loss_dict: 0.3803
  VanillaPipeline.get_eval_image_metrics_and_images: 0.0736
  Trainer.eval_iteration: 0.0055

소요: 213.2분  Return code: 0

Gaussian 분포:
  N=10

In [ ]:
import subprocess, os, re, sys, time
from pathlib import Path
from IPython.display import clear_output

os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']

BASE       = '/content/drive/MyDrive/3dgs_project'
PROC_DIR   = f'{BASE}/processed'
OUTPUT_DIR = f'{BASE}/output'
os.makedirs(f'{OUTPUT_DIR}/logs', exist_ok=True)

EXP_NAME  = f'indoor_scan_{time.strftime("%m%d_%H%M%S")}'
MAX_ITER  = 10000
LOG_EVERY = 50

# mask 파일 존재 확인
mask_dir = Path(PROC_DIR) / 'depth_confidence_masks'
use_masks = mask_dir.exists() and len(list(mask_dir.glob('*.png'))) > 0
print(f'depth_confidence_masks 사용: {use_masks}')

cmd = [
    sys.executable, '-m', 'nerfstudio.scripts.train',
    'ags-mesh',
    '--output-dir', OUTPUT_DIR,
    '--experiment-name', EXP_NAME,
    '--max-num-iterations', str(MAX_ITER),
    '--vis', 'tensorboard',
    '--pipeline.model.use-depth-loss', 'True',
    '--pipeline.model.depth-lambda', '0.05',
    '--pipeline.model.use-normal-loss', 'True',
    '--pipeline.model.use-normal-tv-loss', 'True',
    '--pipeline.model.normal-supervision', 'depth',
    '--pipeline.model.cull-alpha-thresh', '0.005',
    '--pipeline.model.densify-grad-thresh', '0.0001',
    'normal-nerfstudio',
    '--data', PROC_DIR,
    '--load-3D-points', 'True',
    '--load-normals', 'False',
    '--load-depth-confidence-masks', str(use_masks),
    '--downscale-factor', '1',
    '--scene-scale', '75.0',
    '--auto-scale-poses', 'False',
]

print(f'실험: {EXP_NAME}')
pat_step = re.compile(r'^\s*(\d+)\s*\(')
pat_loss = re.compile(r'loss[:\s]+([\d\.eE+\-]+)', re.I)
pat_psnr = re.compile(r'psnr[:\s]+([\d\.]+)', re.I)
last_step, last_loss, last_psnr = 0, '-', '-'
recent_lines = []
start_time = time.time()

def render_status():
    elapsed = time.time() - start_time
    pct = last_step / MAX_ITER * 100
    bar = '#' * int(pct/5) + '.' * (20 - int(pct/5))
    eta_str = time.strftime('%H:%M:%S', time.gmtime(
        elapsed / last_step * (MAX_ITER - last_step))) if last_step > 0 else '--:--:--'
    clear_output(wait=True)
    print(f'--- AGS-Mesh 학습 ---  실험: {EXP_NAME}')
    print(f'  진행: [{bar}] {pct:5.1f}%  ({last_step:>6}/{MAX_ITER})')
    print(f'  경과: {time.strftime("%H:%M:%S", time.gmtime(elapsed))}   ETA: {eta_str}')
    print(f'  Loss: {last_loss:<12}  PSNR: {last_psnr}')
    print('최근 로그:')
    for l in recent_lines[-10:]: print(' ', l)

with open(f'{OUTPUT_DIR}/logs/training_log.txt', 'w') as logf:
    proc = subprocess.Popen(
        cmd, cwd='/content/dn-splatter',
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1)
    for raw_line in proc.stdout:
        for line in raw_line.replace('\r', '\n').split('\n'):
            line = line.strip()
            if not line: continue
            logf.write(line + '\n'); logf.flush()
            recent_lines.append(line)
            if len(recent_lines) > 100: recent_lines.pop(0)
            m = pat_step.search(line)
            if m: last_step = int(m.group(1))
            m = pat_loss.search(line)
            if m: last_loss = m.group(1)
            m = pat_psnr.search(line)
            if m: last_psnr = m.group(1)
            if last_step > 0 and last_step % LOG_EVERY == 0:
                render_status()
    proc.wait()

last_step = MAX_ITER; render_status()
print(f'\n소요: {(time.time()-start_time)/60:.1f}분  Return code: {proc.returncode}')

if proc.returncode == 0:
    import torch, numpy as np
    ckpts = sorted([c for c in Path(OUTPUT_DIR).rglob('*.ckpt')
                    if EXP_NAME in str(c)], key=lambda x: x.stat().st_mtime)
    state = torch.load(str(ckpts[-1]), map_location='cpu', weights_only=False)
    means = state['pipeline']['_model.gauss_params.means'].numpy()
    opacities = torch.sigmoid(
        state['pipeline']['_model.gauss_params.opacities']).numpy().flatten()
    print(f'\nGaussian 분포:')
    print(f'  N={len(means):,}')
    print(f'  X: [{means[:,0].min():.1f}, {means[:,0].max():.1f}]  range={np.ptp(means[:,0]):.1f}m')
    print(f'  Y: [{means[:,1].min():.1f}, {means[:,1].max():.1f}]  range={np.ptp(means[:,1]):.1f}m')
    print(f'  Z: [{means[:,2].min():.1f}, {means[:,2].max():.1f}]  range={np.ptp(means[:,2]):.1f}m')
    print(f'  opacity>0.5: {(opacities>0.5).sum():,} ({(opacities>0.5).mean()*100:.1f}%)')

--- AGS-Mesh 학습 ---  실험: indoor_scan_0527_075043
  진행: [####################] 100.0%  ( 10000/10000)
  경과: 00:38:38   ETA: 00:00:00
  Loss: -             PSNR: -
최근 로그:
  │   Config File          │ /content/drive/MyDrive/3dgs_project/output/indoor_scan_0527_075043/ags-mesh/2026-05-27_0…   │
  │   Checkpoint Directory │ /content/drive/MyDrive/3dgs_project/output/indoor_scan_0527_075043/ags-mesh/2026-05-27_0…   │
  │                        ╵                                                                                             │
  ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
  Printing profiling stats, from longest to shortest duration in seconds
  VanillaPipeline.get_eval_loss_dict: 0.6021
  Trainer.train_iteration: 0.2114
  VanillaPipeline.get_train_loss_dict: 0.1870
  VanillaPipeline.get_eval_image_metrics_and_images: 0.0704
  Trainer.eval_iteration: 0.0014

소요: 38.6분  Return code: 0

Gaussian 분포:
  N=149

In [ ]:
import subprocess, os, re, sys, time
from pathlib import Path
from IPython.display import clear_output

os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']

BASE       = '/content/drive/MyDrive/3dgs_project'
PROC_DIR   = f'{BASE}/processed'
OUTPUT_DIR = f'{BASE}/output'

EXP_NAME  = f'exp_splatfacto_{time.strftime("%m%d_%H%M%S")}'
MAX_ITER  = 5000  # 진단용 짧게
LOG_EVERY = 100

cmd = [
    sys.executable, '-m', 'nerfstudio.scripts.train',
    'splatfacto',
    '--output-dir', OUTPUT_DIR,
    '--experiment-name', EXP_NAME,
    '--max-num-iterations', str(MAX_ITER),
    '--vis', 'tensorboard',
    'nerfstudio-data',
    '--data', PROC_DIR,
    '--downscale-factor', '1',
    '--auto-scale-poses', 'False',
    '--scene-scale', '75.0',
]

print(f'실험: {EXP_NAME}  (splatfacto RGB baseline, 진단용)')
pat_step = re.compile(r'^\s*(\d+)\s*\(')
pat_loss = re.compile(r'loss[:\s]+([\d\.eE+\-]+)', re.I)
last_step, last_loss = 0, '-'
recent_lines = []
start_time = time.time()

def render_status():
    elapsed = time.time() - start_time
    pct = last_step / MAX_ITER * 100
    bar = '#' * int(pct/5) + '.' * (20 - int(pct/5))
    clear_output(wait=True)
    print(f'--- splatfacto ---  실험: {EXP_NAME}')
    print(f'  진행: [{bar}] {pct:5.1f}%  ({last_step:>5}/{MAX_ITER})')
    print(f'  경과: {time.strftime("%H:%M:%S", time.gmtime(elapsed))}')
    print(f'  Loss: {last_loss}')
    for l in recent_lines[-10:]: print(' ', l)

proc = subprocess.Popen(
    cmd, cwd='/content/dn-splatter',
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1)
for raw_line in proc.stdout:
    for line in raw_line.replace('\r', '\n').split('\n'):
        line = line.strip()
        if not line: continue
        recent_lines.append(line)
        if len(recent_lines) > 100: recent_lines.pop(0)
        m = pat_step.search(line)
        if m: last_step = int(m.group(1))
        m = pat_loss.search(line)
        if m: last_loss = m.group(1)
        if last_step > 0 and last_step % LOG_EVERY == 0:
            render_status()
proc.wait()
last_step = MAX_ITER; render_status()
print(f'\nReturn code: {proc.returncode}')

if proc.returncode == 0:
    import torch, numpy as np
    ckpts = sorted([c for c in Path(OUTPUT_DIR).rglob('*.ckpt')
                    if EXP_NAME in str(c)], key=lambda x: x.stat().st_mtime)
    if ckpts:
        state = torch.load(str(ckpts[-1]), map_location='cpu', weights_only=False)
        means = state['pipeline']['_model.gauss_params.means'].numpy()
        print(f'\nGaussian N={len(means):,}')
        print(f'X: [{means[:,0].min():.1f}, {means[:,0].max():.1f}]  range={np.ptp(means[:,0]):.1f}m')
        print(f'Y: [{means[:,1].min():.1f}, {means[:,1].max():.1f}]  range={np.ptp(means[:,1]):.1f}m')
        print(f'Z: [{means[:,2].min():.1f}, {means[:,2].max():.1f}]  range={np.ptp(means[:,2]):.1f}m')

--- splatfacto ---  실험: exp_splatfacto_0527_085846
  진행: [####################] 100.0%  ( 5000/5000)
  경과: 00:04:13
  Loss: -
  │   Checkpoint Directory │ /content/drive/MyDrive/3dgs_project/output/exp_splatfacto_0527_085846/splatfacto/2026-05…   │
  │                        ╵                                                                                             │
  ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
  Printing profiling stats, from longest to shortest duration in seconds
  VanillaPipeline.get_average_eval_image_metrics: 3.6667
  VanillaPipeline.get_average_image_metrics: 3.3338
  VanillaPipeline.get_eval_image_metrics_and_images: 0.2812
  Trainer.train_iteration: 0.0244
  VanillaPipeline.get_train_loss_dict: 0.0188
  Trainer.eval_iteration: 0.0061

Return code: 0

Gaussian N=459,966
X: [-30.3, 48.8]  range=79.1m
Y: [-31.8, 30.2]  range=62.0m
Z: [-25.8, 15.9]  range=41.8m


In [ ]:
import subprocess, os, re, sys, time
from pathlib import Path
from IPython.display import clear_output

os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']

BASE       = '/content/drive/MyDrive/3dgs_project'
PROC_DIR   = f'{BASE}/processed'
OUTPUT_DIR = f'{BASE}/output'
os.makedirs(f'{OUTPUT_DIR}/logs', exist_ok=True)

EXP_NAME  = f'exp_10k_{time.strftime("%m%d_%H%M%S")}'
MAX_ITER  = 10000
LOG_EVERY = 100

cmd = [
    sys.executable, '-m', 'nerfstudio.scripts.train',
    'dn-splatter',
    '--output-dir', OUTPUT_DIR,
    '--experiment-name', EXP_NAME,
    '--max-num-iterations', str(MAX_ITER),
    '--vis', 'tensorboard',
    '--pipeline.model.use-depth-loss', 'False',
    '--pipeline.model.use-normal-loss', 'True',
    '--pipeline.model.use-normal-tv-loss', 'True',
    '--pipeline.model.normal-supervision', 'depth',
    'normal-nerfstudio',
    '--data', PROC_DIR,
    '--load-3D-points', 'True',
    '--load-normals', 'False',
    '--load-depth-confidence-masks', 'False',
    '--downscale-factor', '1',
    '--scene-scale', '75.0',
    '--auto-scale-poses', 'False',
]

print(f'실험: {EXP_NAME}')
print(f'depth-lambda=0.05  scene-scale=75  10000 iter')
pat_step = re.compile(r'^\s*(\d+)\s*\(')
pat_loss = re.compile(r'loss[:\s]+([\d\.eE+\-]+)', re.I)
pat_psnr = re.compile(r'psnr[:\s]+([\d\.]+)', re.I)
last_step, last_loss, last_psnr = 0, '-', '-'
recent_lines = []
start_time = time.time()

def render_status():
    elapsed = time.time() - start_time
    pct = last_step / MAX_ITER * 100
    bar = '#' * int(pct/5) + '.' * (20 - int(pct/5))
    eta_str = time.strftime('%H:%M:%S', time.gmtime(
        elapsed / last_step * (MAX_ITER - last_step))) if last_step > 0 else '--:--:--'
    clear_output(wait=True)
    print(f'--- AGS-Mesh ---  실험: {EXP_NAME}')
    print(f'  진행: [{bar}] {pct:5.1f}%  ({last_step:>6}/{MAX_ITER})')
    print(f'  경과: {time.strftime("%H:%M:%S", time.gmtime(elapsed))}   ETA: {eta_str}')
    print(f'  Loss: {last_loss:<12}  PSNR: {last_psnr}')
    print('최근 로그:')
    for l in recent_lines[-10:]: print(' ', l)

with open(f'{OUTPUT_DIR}/logs/training_log.txt', 'w') as logf:
    proc = subprocess.Popen(
        cmd, cwd='/content/dn-splatter',
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1)
    for raw_line in proc.stdout:
        for line in raw_line.replace('\r', '\n').split('\n'):
            line = line.strip()
            if not line: continue
            logf.write(line + '\n'); logf.flush()
            recent_lines.append(line)
            if len(recent_lines) > 100: recent_lines.pop(0)
            m = pat_step.search(line)
            if m: last_step = int(m.group(1))
            m = pat_loss.search(line)
            if m: last_loss = m.group(1)
            m = pat_psnr.search(line)
            if m: last_psnr = m.group(1)
            if last_step > 0 and last_step % LOG_EVERY == 0:
                render_status()
    proc.wait()

last_step = MAX_ITER; render_status()
print(f'\n소요: {(time.time()-start_time)/60:.1f}분  Return code: {proc.returncode}')

if proc.returncode == 0:
    import torch, numpy as np
    ckpts = sorted([c for c in Path(OUTPUT_DIR).rglob('*.ckpt')
                    if EXP_NAME in str(c)], key=lambda x: x.stat().st_mtime)
    state = torch.load(str(ckpts[-1]), map_location='cpu', weights_only=False)
    means = state['pipeline']['_model.gauss_params.means'].numpy()
    opacities = torch.sigmoid(
        state['pipeline']['_model.gauss_params.opacities']).numpy().flatten()
    print(f'\nGaussian 분포:')
    print(f'  N={len(means):,}')
    print(f'  X: [{means[:,0].min():.1f}, {means[:,0].max():.1f}]  range={np.ptp(means[:,0]):.1f}m')
    print(f'  Y: [{means[:,1].min():.1f}, {means[:,1].max():.1f}]  range={np.ptp(means[:,1]):.1f}m')
    print(f'  Z: [{means[:,2].min():.1f}, {means[:,2].max():.1f}]  range={np.ptp(means[:,2]):.1f}m')
    print(f'  opacity>0.5: {(opacities>0.5).sum():,} ({(opacities>0.5).mean()*100:.1f}%)')

--- AGS-Mesh ---  실험: exp_10k_0527_145155
  진행: [####################] 100.0%  ( 10000/10000)
  경과: 00:38:28   ETA: 00:00:00
  Loss: -             PSNR: -
최근 로그:
  │   Config File          │ /content/drive/MyDrive/3dgs_project/output/exp_10k_0527_145155/dn-splatter/2026-05-27_14…   │
  │   Checkpoint Directory │ /content/drive/MyDrive/3dgs_project/output/exp_10k_0527_145155/dn-splatter/2026-05-27_14…   │
  │                        ╵                                                                                             │
  ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
  Printing profiling stats, from longest to shortest duration in seconds
  VanillaPipeline.get_eval_loss_dict: 0.6089
  Trainer.train_iteration: 0.2124
  VanillaPipeline.get_train_loss_dict: 0.1901
  VanillaPipeline.get_eval_image_metrics_and_images: 0.0733
  Trainer.eval_iteration: 0.0014

소요: 38.5분  Return code: 0

Gaussian 분포:
  N=52,050
  X

In [ ]:
import subprocess, sys

for method in ['ags-mesh', 'dn-splatter']:
    print(f'\n===== {method} =====')
    result = subprocess.run(
        [sys.executable, '-m', 'nerfstudio.scripts.train', method, '--help'],
        capture_output=True, text=True
    )
    for line in result.stdout.split('\n'):
        low = line.lower()
        if any(k in low for k in ['scale', 'cull', 'prune', 'split', 'densify', 'gauss-ratio', 'max-gauss']):
            print(line)


===== ags-mesh =====
│ --use-grad-scaler {True,False}                                               │
│       Use gradient scaler even if the automatic mixed precision is disabled. │
│ --viewer.camera-frustum-scale FLOAT                                          │
│       Scale for the camera frustums in the viewer. (default: 0.1)            │
│ --pipeline.datamanager.camera-res-scale-factor FLOAT                         │
│       Rescale cameras (default: 1.0)                                         │
│       period of steps where gaussians are culled and densified (default:     │
│ --pipeline.model.num-downscales INT                                          │
│ --pipeline.model.cull-alpha-thresh FLOAT                                     │
│       threshold of opacity for culling gaussians. One can set it to a lower  │
│ --pipeline.model.cull-scale-thresh FLOAT                                     │
│       threshold of scale for culling huge gaussians (default: 0.5)           │
│ --pi

In [ ]:
import subprocess, sys, os
from pathlib import Path

os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']

BASE       = '/content/drive/MyDrive/3dgs_project'
OUTPUT_DIR = f'{BASE}/output'

configs = sorted(
    [c for c in Path(OUTPUT_DIR).rglob('config.yml')
     if 'splatfacto' in str(c) and (c.parent / 'nerfstudio_models').exists()],
    key=lambda x: x.stat().st_mtime
)
CONFIG = str(configs[-1])
CKPT_DIR = str(configs[-1].parent)
print(f'config: {CONFIG}')

proc = subprocess.run([
    sys.executable, '-m', 'nerfstudio.scripts.exporter',
    'gaussian-splat',
    '--load-config', CONFIG,
    '--output-dir', CKPT_DIR
], capture_output=True, text=True)

print(proc.stdout[-500:])
if proc.returncode != 0:
    print(proc.stderr[-300:])

for f in Path(CKPT_DIR).rglob('*.ply'):
    print(f'{f.stat().st_size/1e6:.1f} MB  {f}')

config: /content/drive/MyDrive/3dgs_project/output/exp_splatfacto_0527_085846/splatfacto/2026-05-27_085902/config.yml
Train dataset has over 500 images, overriding cache_images to cpu
Loading latest checkpoint from load_dir
✅ Done loading checkpoint from 
/content/drive/MyDrive/3dgs_project/output/exp_splatfacto_0527_085846/splatfacto/2026-05-27_085902/nerfstudio_models/ste
p-000004999.ckpt

114.1 MB  /content/drive/MyDrive/3dgs_project/output/exp_splatfacto_0527_085846/splatfacto/2026-05-27_085902/splat.ply


In [ ]:
import subprocess, os, re, sys, time
from pathlib import Path
from IPython.display import clear_output

os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']

BASE       = '/content/drive/MyDrive/3dgs_project'
PROC_DIR   = f'{BASE}/processed'
OUTPUT_DIR = f'{BASE}/output'
os.makedirs(f'{OUTPUT_DIR}/logs', exist_ok=True)

EXP_NAME  = f'indoor_scan_{time.strftime("%m%d_%H%M%S")}'
MAX_ITER  = 10000
LOG_EVERY = 50

cmd = [
    sys.executable, '-m', 'nerfstudio.scripts.train',
    'dn-splatter',
    '--output-dir', OUTPUT_DIR,
    '--experiment-name', EXP_NAME,
    '--max-num-iterations', str(MAX_ITER),
    '--vis', 'tensorboard',
    '--pipeline.model.use-depth-loss', 'True',
    '--pipeline.model.depth-lambda', '0.2',
    '--pipeline.model.use-normal-loss', 'True',
    '--pipeline.model.use-normal-tv-loss', 'True',
    '--pipeline.model.normal-supervision', 'depth',
    '--pipeline.model.cull-alpha-thresh', '0.005',
    '--pipeline.model.cull-scale-thresh', '0.5',
    '--pipeline.model.use-scale-regularization', 'True',
    '--pipeline.model.max-gauss-ratio', '5.0',
    'normal-nerfstudio',
    '--data', PROC_DIR,
    '--load-3D-points', 'True',
    '--load-normals', 'False',
    '--load-depth-confidence-masks', 'False',
    '--downscale-factor', '1',
    '--scene-scale', '75.0',
    '--auto-scale-poses', 'False',
]

print(f'실험: {EXP_NAME}')
pat_step = re.compile(r'^\s*(\d+)\s*\(')
pat_loss = re.compile(r'loss[:\s]+([\d\.eE+\-]+)', re.I)
pat_psnr = re.compile(r'psnr[:\s]+([\d\.]+)', re.I)
last_step, last_loss, last_psnr = 0, '-', '-'
recent_lines = []
start_time = time.time()

def render_status():
    elapsed = time.time() - start_time
    pct = last_step / MAX_ITER * 100
    bar = '#' * int(pct/5) + '.' * (20 - int(pct/5))
    eta_str = time.strftime('%H:%M:%S', time.gmtime(
        elapsed / last_step * (MAX_ITER - last_step))) if last_step > 0 else '--:--:--'
    clear_output(wait=True)
    print(f'--- DN-Splatter ---  실험: {EXP_NAME}')
    print(f'  진행: [{bar}] {pct:5.1f}%  ({last_step:>6}/{MAX_ITER})')
    print(f'  경과: {time.strftime("%H:%M:%S", time.gmtime(elapsed))}   ETA: {eta_str}')
    print(f'  Loss: {last_loss:<12}  PSNR: {last_psnr}')
    print('최근 로그:')
    for l in recent_lines[-10:]: print(' ', l)

with open(f'{OUTPUT_DIR}/logs/training_log.txt', 'w') as logf:
    proc = subprocess.Popen(
        cmd, cwd='/content/dn-splatter',
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1)
    for raw_line in proc.stdout:
        for line in raw_line.replace('\r', '\n').split('\n'):
            line = line.strip()
            if not line: continue
            logf.write(line + '\n'); logf.flush()
            recent_lines.append(line)
            if len(recent_lines) > 100: recent_lines.pop(0)
            m = pat_step.search(line)
            if m: last_step = int(m.group(1))
            m = pat_loss.search(line)
            if m: last_loss = m.group(1)
            m = pat_psnr.search(line)
            if m: last_psnr = m.group(1)
            if last_step > 0 and last_step % LOG_EVERY == 0:
                render_status()
    proc.wait()

last_step = MAX_ITER; render_status()
print(f'\n소요: {(time.time()-start_time)/60:.1f}분  Return code: {proc.returncode}')

if proc.returncode == 0:
    import torch, numpy as np
    ckpts = sorted([c for c in Path(OUTPUT_DIR).rglob('*.ckpt')
                    if EXP_NAME in str(c)], key=lambda x: x.stat().st_mtime)
    state = torch.load(str(ckpts[-1]), map_location='cpu', weights_only=False)
    means = state['pipeline']['_model.gauss_params.means'].numpy()
    opacities = torch.sigmoid(
        state['pipeline']['_model.gauss_params.opacities']).numpy().flatten()
    print(f'\nGaussian 분포:')
    print(f'  N={len(means):,}')
    print(f'  X: [{means[:,0].min():.1f}, {means[:,0].max():.1f}]  range={np.ptp(means[:,0]):.1f}m')
    print(f'  Y: [{means[:,1].min():.1f}, {means[:,1].max():.1f}]  range={np.ptp(means[:,1]):.1f}m')
    print(f'  Z: [{means[:,2].min():.1f}, {means[:,2].max():.1f}]  range={np.ptp(means[:,2]):.1f}m')
    print(f'  opacity>0.5: {(opacities>0.5).sum():,} ({(opacities>0.5).mean()*100:.1f}%)')

--- DN-Splatter ---  실험: indoor_scan_0527_172848
  진행: [####################] 100.0%  ( 10000/10000)
  경과: 00:40:30   ETA: 00:00:00
  Loss: -             PSNR: -
최근 로그:
  │   Config File          │ /content/drive/MyDrive/3dgs_project/output/indoor_scan_0527_172848/dn-splatter/2026-05-2…   │
  │   Checkpoint Directory │ /content/drive/MyDrive/3dgs_project/output/indoor_scan_0527_172848/dn-splatter/2026-05-2…   │
  │                        ╵                                                                                             │
  ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
  Printing profiling stats, from longest to shortest duration in seconds
  VanillaPipeline.get_eval_loss_dict: 0.6043
  Trainer.train_iteration: 0.2225
  VanillaPipeline.get_train_loss_dict: 0.1964
  VanillaPipeline.get_eval_image_metrics_and_images: 0.0719
  Trainer.eval_iteration: 0.0014

소요: 40.5분  Return code: 0

Gaussian 분포:
  N=29,

In [ ]:
import subprocess, sys, os
from pathlib import Path

os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']

BASE       = '/content/drive/MyDrive/3dgs_project'
OUTPUT_DIR = f'{BASE}/output'

# 가장 좋았던 실험 config 선택
configs = sorted(
    [c for c in Path(OUTPUT_DIR).rglob('config.yml')
     if (c.parent / 'nerfstudio_models').exists()],
    key=lambda x: x.stat().st_mtime
)
CONFIG = str(configs[-1])
CKPT_DIR = str(configs[-1].parent)
print(f'config: {CONFIG}')

proc = subprocess.run([
    sys.executable, '-m', 'nerfstudio.scripts.exporter',
    'gaussian-splat',
    '--load-config', CONFIG,
    '--output-dir', CKPT_DIR
], capture_output=True, text=True)
print(proc.stdout[-500:])

for f in Path(CKPT_DIR).rglob('*.ply'):
    print(f'{f.stat().st_size/1e6:.1f} MB  {f}')

config: /content/drive/MyDrive/3dgs_project/output/indoor_scan_0527_172848/dn-splatter/2026-05-27_172906/config.yml
Train dataset has over 500 images, overriding cache_images to cpu
[18:18:16] Number of initial seed points 354000                                                          dn_model.py:136
Initialising Gaussian normals from intial seed points
Loading latest checkpoint from load_dir
✅ Done loading checkpoint from 
/content/drive/MyDrive/3dgs_project/output/indoor_scan_0527_172848/dn-splatter/2026-05-27_172906/nerfstudio_models/step-
000009999.ckpt

7.4 MB  /content/drive/MyDrive/3dgs_project/output/indoor_scan_0527_172848/dn-splatter/2026-05-27_172906/splat.ply


In [ ]:
import torch, numpy as np
from pathlib import Path

BASE = '/content/drive/MyDrive/3dgs_project'

# 모든 체크포인트 N과 opacity 확인
ckpts = sorted(Path(f'{BASE}/output').rglob('step-000029999.ckpt'),
               key=lambda x: x.stat().st_mtime)

for ckpt in ckpts:
    try:
        state = torch.load(str(ckpt), map_location='cpu', weights_only=False)
        means = state['pipeline']['_model.gauss_params.means'].numpy()
        opacities = torch.sigmoid(
            state['pipeline']['_model.gauss_params.opacities']).numpy().flatten()
        exp = ckpt.parent.parent.parent.name
        print(f'{exp}  N={len(means):,}  opacity>0.5={( opacities>0.5).mean()*100:.1f}%  '
              f'Z_range={np.ptp(means[:,2]):.1f}m')
    except:
        pass

dn-splatter  N=605,498  opacity>0.5=87.6%  Z_range=10.0m
dn-splatter  N=653,530  opacity>0.5=59.1%  Z_range=10.0m
dn-splatter  N=269,630  opacity>0.5=57.4%  Z_range=10.3m
ags-mesh  N=441,197  opacity>0.5=2.1%  Z_range=15.6m
ags-mesh  N=253,999  opacity>0.5=92.7%  Z_range=7.8m
ags-mesh  N=273,697  opacity>0.5=90.4%  Z_range=8.3m
ags-mesh  N=13,861  opacity>0.5=95.8%  Z_range=6.1m
ags-mesh  N=13,748  opacity>0.5=95.7%  Z_range=7.4m
ags-mesh  N=102,325  opacity>0.5=68.8%  Z_range=9.0m


In [ ]:
import json, numpy as np
from pathlib import Path

PROC_DIR = '/content/drive/MyDrive/3dgs_project/processed'

with open(f'{PROC_DIR}/transforms.json') as f:
    tf = json.load(f)

positions = np.array([np.array(f['transform_matrix'])[:3,3] for f in tf['frames']])

print('=== 카메라 궤적 기반 공간 추정 ===')
print(f'X: [{positions[:,0].min():.2f}, {positions[:,0].max():.2f}]  range={np.ptp(positions[:,0]):.2f}m')
print(f'Y: [{positions[:,1].min():.2f}, {positions[:,1].max():.2f}]  range={np.ptp(positions[:,1]):.2f}m')
print(f'Z: [{positions[:,2].min():.2f}, {positions[:,2].max():.2f}]  range={np.ptp(positions[:,2]):.2f}m')

total_range = np.ptp(positions, axis=0)
diagonal = np.linalg.norm(total_range)
print(f'\n공간 대각선 길이 (최대 span): {diagonal:.2f}m')

# 이동 경로 총 길이
diffs = np.linalg.norm(np.diff(positions, axis=0), axis=1)
total_path = diffs.sum()
print(f'카메라 이동 총 경로: {total_path:.2f}m')

# 권장 scene-scale (공간 대각선의 1.5배)
recommended = diagonal * 1.5
print(f'\n권장 scene-scale: {recommended:.0f}  (대각선 {diagonal:.1f}m × 1.5)')
print(f'현재 설정: 75.0')

=== 카메라 궤적 기반 공간 추정 ===
X: [-19.64, 41.64]  range=61.28m
Y: [-0.45, 0.37]  range=0.82m
Z: [-6.52, 29.42]  range=35.94m

공간 대각선 길이 (최대 span): 71.04m
카메라 이동 총 경로: 306.88m

권장 scene-scale: 107  (대각선 71.0m × 1.5)
현재 설정: 75.0


In [ ]:
import json, numpy as np, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

BASE     = '/content/drive/MyDrive/3dgs_project'
PROC_DIR = f'{BASE}/processed'
LOG_DIR  = f'{PROC_DIR}/logs'

with open(f'{PROC_DIR}/transforms.json') as f:
    tf = json.load(f)

positions = np.array([np.array(f['transform_matrix'])[:3,3] for f in tf['frames']])

# 1. loop error
start = positions[0]
end   = positions[-1]
loop_error = np.linalg.norm(end - start)
print(f'시작점: {start.round(3)}')
print(f'끝점:   {end.round(3)}')
print(f'loop error: {loop_error:.3f}m')

if loop_error < 1.0:
    print('✅ 양호')
elif loop_error < 3.0:
    print('⚠️  사용 가능하나 mesh 흔들릴 수 있음')
elif loop_error < 10.0:
    print('⚠️  장거리 학습에서 터질 가능성 있음')
else:
    print('❌ pose 문제 강함')

# 2. trajectory top-view
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 색상 = 시간순
sc = axes[0].scatter(positions[:,0], positions[:,2],
                     c=np.arange(len(positions)), cmap='viridis', s=3)
axes[0].plot(positions[0,0], positions[0,2], 'go', ms=10, label='start')
axes[0].plot(positions[-1,0], positions[-1,2], 'rs', ms=10, label='end')
plt.colorbar(sc, ax=axes[0], label='frame index')
axes[0].set_title('Top view (X-Z) 색상=시간순')
axes[0].set_aspect('equal'); axes[0].legend()

# Z값 시간변화 (복도 이동 패턴)
axes[1].plot(positions[:,2], 'b-', lw=0.5)
axes[1].set_title('Z값 시간변화')
axes[1].set_xlabel('frame index'); axes[1].set_ylabel('Z (m)')

plt.tight_layout()
plt.savefig(f'{LOG_DIR}/loop_check.png', dpi=120); plt.close()
print('저장: logs/loop_check.png')

# 3. 인접 이동거리
diffs = np.linalg.norm(np.diff(positions, axis=0), axis=1)
print(f'\n인접 이동거리: mean={diffs.mean():.4f}m  max={diffs.max():.4f}m')
print(f'점프(>0.5m): {(diffs>0.5).sum()}건')

시작점: [0.196 0.162 4.244]
끝점:   [0.976 0.148 4.525]
loop error: 0.828m
✅ 양호


/tmp/ipykernel_2729/998206813.py:49: UserWarning: Glyph 49353 (\N{HANGUL SYLLABLE SAEG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2729/998206813.py:49: UserWarning: Glyph 49345 (\N{HANGUL SYLLABLE SANG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2729/998206813.py:49: UserWarning: Glyph 49884 (\N{HANGUL SYLLABLE SI}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2729/998206813.py:49: UserWarning: Glyph 44036 (\N{HANGUL SYLLABLE GAN}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2729/998206813.py:49: UserWarning: Glyph 49692 (\N{HANGUL SYLLABLE SUN}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2729/998206813.py:49: UserWarning: Glyph 44050 (\N{HANGUL SYLLABLE GABS}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2729/998206813.py:49: UserWarning: Glyph 48320 (\N{HANGUL SYLLABLE BYEON}) missing from font(s) DejaVu Sans.
  plt.tight_layo

저장: logs/loop_check.png

인접 이동거리: mean=0.1738m  max=0.3189m
점프(>0.5m): 0건


In [ ]:
import json
from pathlib import Path

PROC_DIR = '/content/drive/MyDrive/3dgs_project/processed'

with open(f'{PROC_DIR}/transforms.json') as f:
    tf = json.load(f)

print(f'총 프레임: {len(tf["frames"])}')
print(f'depth 포함: {sum(1 for f in tf["frames"] if "depth_file_path" in f)}')
print(f'ply_file_path: {tf.get("ply_file_path", "없음")}')

# pointcloud 확인
ply = Path(f'{PROC_DIR}/sparse_pointcloud.ply')
if ply.exists():
    print(f'pointcloud: {ply.stat().st_size/1e6:.1f} MB')
else:
    print('pointcloud 없음 → Cell 2e 실행 필요')

총 프레임: 1767
depth 포함: 1767
ply_file_path: sparse_pointcloud.ply
pointcloud: 5.3 MB


In [ ]:
# export
import subprocess, sys, os
from pathlib import Path

os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']

BASE       = '/content/drive/MyDrive/3dgs_project'
OUTPUT_DIR = f'{BASE}/output'

configs = sorted(Path(OUTPUT_DIR).rglob('config.yml'),
                 key=lambda x: x.stat().st_mtime)
CKPT_DIR = str(configs[-1].parent)
print(f'체크포인트: {CKPT_DIR}')

proc = subprocess.run([
    sys.executable, '-m', 'nerfstudio.scripts.exporter',
    'gaussian-splat',
    '--load-config', f'{CKPT_DIR}/config.yml',
    '--output-dir', CKPT_DIR
], capture_output=True, text=True)

print(proc.stdout[-1000:])
if proc.returncode != 0:
    print('STDERR:', proc.stderr[-500:])

ply_files = list(Path(CKPT_DIR).rglob('*.ply'))
for f in ply_files:
    print(f'{f.stat().st_size/1e6:.1f} MB  {f}')

In [ ]:
import torch, numpy as np
from pathlib import Path

BASE = '/content/drive/MyDrive/3dgs_project'
ckpts = sorted(Path(f'{BASE}/output').rglob('*.ckpt'),
               key=lambda x: x.stat().st_mtime)
state = torch.load(str(ckpts[-1]), map_location='cpu', weights_only=False)
means = state['pipeline']['_model.gauss_params.means'].numpy()

print(f'Gaussian 수: {len(means):,}')
print(f'X: [{means[:,0].min():.2f}, {means[:,0].max():.2f}]  range={np.ptp(means[:,0]):.2f}')
print(f'Y: [{means[:,1].min():.2f}, {means[:,1].max():.2f}]  range={np.ptp(means[:,1]):.2f}')
print(f'Z: [{means[:,2].min():.2f}, {means[:,2].max():.2f}]  range={np.ptp(means[:,2]):.2f}')

In [ ]:
import torch, numpy as np
from pathlib import Path

BASE = '/content/drive/MyDrive/3dgs_project'

# 모든 ckpt의 Gaussian 범위 확인
ckpts = sorted(Path(f'{BASE}/output').rglob('*.ckpt'),
               key=lambda x: x.stat().st_mtime)

for ckpt in ckpts[-5:]:
    try:
        state = torch.load(str(ckpt), map_location='cpu', weights_only=False)
        means = state['pipeline']['_model.gauss_params.means'].numpy()
        z_range = np.ptp(means[:,2])
        print(f'{ckpt.parent.parent.name}  step={ckpt.stem}  '
              f'N={len(means):,}  Z_range={z_range:.1f}m')
    except:
        print(f'{ckpt.name} 로드 실패')

In [ ]:
import json
from pathlib import Path

BASE = '/content/drive/MyDrive/3dgs_project'
transforms_files = sorted(
    Path(f'{BASE}/output').rglob('dataparser_transforms.json'),
    key=lambda x: x.stat().st_mtime
)
data = json.loads(transforms_files[-1].read_text())
print(f'scale: {data["scale"]}')
print(f'파일: {transforms_files[-1]}')

In [ ]:
import struct
import numpy as np
from pathlib import Path

ply = Path('/content/drive/MyDrive/3dgs_project/processed/sparse_pointcloud.ply')
print(f'크기: {ply.stat().st_size/1e6:.1f} MB')

# 헤더 파싱해서 포인트 수 확인
with open(ply, 'rb') as f:
    header = b''
    while True:
        line = f.readline()
        header += line
        if line.strip() == b'end_header':
            break
    header_text = header.decode()
    n_pts = int([l for l in header_text.split('\n') if 'element vertex' in l][0].split()[-1])
    print(f'포인트 수: {n_pts:,}')

    # 샘플링해서 범위 확인
    data = f.read()

pts = np.frombuffer(data, dtype=np.dtype([
    ('x','f4'),('y','f4'),('z','f4'),
    ('r','u1'),('g','u1'),('b','u1')
]))
print(f'X: [{pts["x"].min():.2f}, {pts["x"].max():.2f}]  range={np.ptp(pts["x"]):.2f}m')
print(f'Y: [{pts["y"].min():.2f}, {pts["y"].max():.2f}]  range={np.ptp(pts["y"]):.2f}m')
print(f'Z: [{pts["z"].min():.2f}, {pts["z"].max():.2f}]  range={np.ptp(pts["z"]):.2f}m')

크기: 2.0 MB
포인트 수: 132,000
X: [-12.08, 10.54]  range=22.62m
Y: [-1.64, 2.35]  range=3.99m
Z: [-10.46, 52.74]  range=63.20m


In [ ]:
from pathlib import Path

log = Path('/content/drive/MyDrive/3dgs_project/output/logs/03_training_log.txt').read_text()
lines = log.split('\n')

# PSNR 있는 줄 찾기
psnr_lines = [l for l in lines if 'psnr' in l.lower()]
print(f'PSNR 로그 수: {len(psnr_lines)}')
print('\n'.join(psnr_lines[:5]))
print('...')
print('\n'.join(psnr_lines[-5:]))

# loss 있는 줄 샘플
loss_lines = [l for l in lines if 'loss' in l.lower() and any(c.isdigit() for c in l)]
print(f'\nLoss 로그 수: {len(loss_lines)}')
print('\n'.join(loss_lines[:5]))
print('...')
print('\n'.join(loss_lines[-5:]))

In [ ]:
import json, numpy as np
from pathlib import Path

with open('/content/drive/MyDrive/3dgs_project/processed/transforms.json') as f:
    tf = json.load(f)

positions = np.array([np.array(f['transform_matrix'])[:3,3] for f in tf['frames']])

# 시작/중간/끝 포즈 확인
for i, label in [(0,'시작'), (len(tf['frames'])//2,'중간'), (-1,'끝')]:
    p = positions[i]
    print(f'{label}: X={p[0]:.2f}  Y={p[1]:.2f}  Z={p[2]:.2f}')

# 이동 방향 확인 (Z가 주 이동축이어야 함)
print(f'\nZ 방향 이동: {positions[-1,2] - positions[0,2]:.2f}m')
print(f'X 방향 이동: {positions[-1,0] - positions[0,0]:.2f}m')

# forward 벡터 확인 (처음/끝)
for i, label in [(0,'시작'), (-1,'끝')]:
    c2w = np.array(tf['frames'][i]['transform_matrix'])
    print(f'\n{label} forward: {c2w[:3,2].round(3)}')
    print(f'{label} up:      {c2w[:3,1].round(3)}')

In [ ]:
import pandas as pd, cv2

odo = pd.read_csv('/content/drive/MyDrive/3dgs_project/input/odometry.csv')
odo.columns = [c.strip().lower() for c in odo.columns]  # 소문자 변환
cap = cv2.VideoCapture('/content/drive/MyDrive/3dgs_project/input/rgb.mp4')

n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
src_fps   = cap.get(cv2.CAP_PROP_FPS)
cap.release()

print(f'video frames: {n_frames}  fps: {src_fps}')
print(f'odometry rows: {len(odo)}')
print(f'odometry frame 범위: {int(odo["frame"].min())} ~ {int(odo["frame"].max())}')
print(f'\n처음 5행:')
print(odo[['frame','x','y','z']].head())
print(f'\n마지막 5행:')
print(odo[['frame','x','y','z']].tail())

# stride=6 기준으로 실제 매핑되는 frame 확인
stride = 6
sample_video_frames = [0, 6, 12, 18, 24, 30]
print(f'\nvideo frame → odometry frame 매핑 샘플:')
odo_by_frame = {int(row['frame']): row for _, row in odo.iterrows()}
for vf in sample_video_frames:
    if vf in odo_by_frame:
        row = odo_by_frame[vf]
        print(f'  video[{vf}] → odo frame {int(row["frame"])}  z={row["z"]:.2f}')
    else:
        closest = min(odo_by_frame.keys(), key=lambda x: abs(x-vf))
        print(f'  video[{vf}] → closest odo frame {closest}')

In [ ]:
import torch, numpy as np
from pathlib import Path

BASE = '/content/drive/MyDrive/3dgs_project'
ckpts = sorted(Path(f'{BASE}/output').rglob('*.ckpt'),
               key=lambda x: x.stat().st_mtime)
state = torch.load(str(ckpts[-1]), map_location='cpu', weights_only=False)

means = state['pipeline']['_model.gauss_params.means'].numpy()
print(f'Gaussian 수: {len(means):,}')
print(f'X: [{means[:,0].min():.2f}, {means[:,0].max():.2f}]  range={np.ptp(means[:,0]):.2f}m')
print(f'Y: [{means[:,1].min():.2f}, {means[:,1].max():.2f}]  range={np.ptp(means[:,1]):.2f}m')
print(f'Z: [{means[:,2].min():.2f}, {means[:,2].max():.2f}]  range={np.ptp(means[:,2]):.2f}m')

# 복도 범위 밖 Gaussian 비율
in_corridor = (
    (means[:,0] > -8) & (means[:,0] < 3) &
    (means[:,1] > -3) & (means[:,1] < 3) &
    (means[:,2] > -10) & (means[:,2] < 52)
)
print(f'\n복도 범위 내 Gaussian: {in_corridor.sum():,} ({in_corridor.mean()*100:.1f}%)')
print(f'복도 범위 밖 Gaussian: {(~in_corridor).sum():,} ({(~in_corridor).mean()*100:.1f}%)')

In [ ]:
import json
from pathlib import Path

BASE = '/content/drive/MyDrive/3dgs_project'

# 가장 최신 학습 폴더의 dataparser_transforms.json
transforms_files = sorted(
    Path(f'{BASE}/output').rglob('dataparser_transforms.json'),
    key=lambda x: x.stat().st_mtime
)
print(f'파일: {transforms_files[-1]}')
data = json.loads(transforms_files[-1].read_text())
print(json.dumps(data, indent=2))

In [ ]:
import json, numpy as np, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

with open('/content/drive/MyDrive/3dgs_project/processed/transforms.json') as f:
    tf = json.load(f)

positions = np.array([np.array(f['transform_matrix'])[:3,3] for f in tf['frames']])

# Z값 변화 (시간순)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].plot(positions[:,2], 'b-', lw=0.8)
axes[0,0].set_title('Z값 시간변화 (복도 이동축)')
axes[0,0].set_xlabel('frame index'); axes[0,0].set_ylabel('Z (m)')

axes[0,1].plot(positions[:,0], positions[:,2], 'b-o', ms=1, lw=0.5)
axes[0,1].set_title('Top view (X-Z)')
axes[0,1].set_aspect('equal')

axes[1,0].plot(positions[:,2], positions[:,1], 'b-o', ms=1, lw=0.5)
axes[1,0].set_title('Side view (Z-Y)')

axes[1,1].plot(range(len(positions)), positions[:,2], 'b-', lw=0.8)
n = len(positions)
axes[1,1].axvline(n//4, color='r', ls='--', label='25%')
axes[1,1].axvline(n//2, color='g', ls='--', label='50%')
axes[1,1].axvline(3*n//4, color='orange', ls='--', label='75%')
axes[1,1].legend(); axes[1,1].set_title('Z 진행 상황')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/3dgs_project/processed/logs/trajectory_detail.png', dpi=120)
plt.close()
print('저장 완료: logs/trajectory_detail.png')

# 구간별 Z 통계
print(f'\n구간별 Z 평균:')
for i in range(4):
    seg = positions[i*n//4:(i+1)*n//4, 2]
    print(f'  {i*25}~{(i+1)*25}%: Z mean={seg.mean():.1f}  max={seg.max():.1f}  min={seg.min():.1f}')

In [ ]:
import numpy as np
from PIL import Image
from pathlib import Path

INPUT_DIR = '/content/drive/MyDrive/3dgs_project/input'
depth_files = sorted(Path(f'{INPUT_DIR}/depth').glob('*.png'))

# 첫 번째 depth 파일 raw 값 확인
d = np.array(Image.open(depth_files[0]))
print(f'dtype: {d.dtype}')
print(f'shape: {d.shape}')
print(f'raw min/max: {d.min()} / {d.max()}')
print(f'nonzero mean: {d[d>0].mean():.1f}')
print(f'\n/1000 변환 시: {d[d>0].mean()/1000:.3f}m (평균)')
print(f'/1000 변환 시 max: {d.max()/1000:.3f}m')

# confidence 확인
conf_files = sorted(Path(f'{INPUT_DIR}/confidence').glob('*.png'))
c = np.array(Image.open(conf_files[0]))
print(f'\nconfidence unique values: {np.unique(c)}')
print(f'confidence=2 비율: {(c==2).mean()*100:.1f}%')

In [ ]:
import json, numpy as np
from pathlib import Path

with open('/content/drive/MyDrive/3dgs_project/processed/transforms.json') as f:
    tf = json.load(f)

positions = np.array([np.array(f['transform_matrix'])[:3,3] for f in tf['frames']])
print(f'X: [{positions[:,0].min():.2f}, {positions[:,0].max():.2f}]  range={np.ptp(positions[:,0]):.2f}m')
print(f'Y: [{positions[:,1].min():.2f}, {positions[:,1].max():.2f}]  range={np.ptp(positions[:,1]):.2f}m')
print(f'Z: [{positions[:,2].min():.2f}, {positions[:,2].max():.2f}]  range={np.ptp(positions[:,2]):.2f}m')

# 카메라가 어느 방향을 보는지 확인 (forward 벡터)
c2w = np.array(tf['frames'][len(tf['frames'])//2]['transform_matrix'])
print(f'\n중간 프레임 forward 벡터: {c2w[:3,2]}')
print(f'중간 프레임 up 벡터: {c2w[:3,1]}')
print(f'카메라 위치: {c2w[:3,3]}')

In [ ]:
import pandas as pd

odo = pd.read_csv('/content/drive/MyDrive/3dgs_project/input/odometry.csv')
odo.columns = [c.strip().lower() for c in odo.columns]
print(odo.head(10).to_string())
print(f'\n전체 행 수: {len(odo)}')

# x,y,z 범위
for col in odo.columns:
    if odo[col].dtype in ['float64', 'int64']:
        print(f'{col}: min={odo[col].min():.4f}  max={odo[col].max():.4f}  range={odo[col].max()-odo[col].min():.4f}')

In [ ]:
# 패치 대상 확인
from pathlib import Path

reg = Path('/content/dn-splatter/dn_splatter/regularization_strategy.py').read_text()
dm  = Path('/content/dn-splatter/dn_splatter/dn_model.py').read_text()

# confidence 관련 라인 전부 출력
print('=== regularization_strategy.py ===')
for i, l in enumerate(reg.split('\n'), 1):
    if 'confidence' in l.lower():
        print(f'  {i}: {l}')

print('\n=== dn_model.py ===')
for i, l in enumerate(dm.split('\n'), 1):
    if 'confidence' in l.lower():
        print(f'  {i}: {l}')

In [ ]:
# ============================================================
# Cell 3-patch: nerfstudio 버그 패치 (학습 전 필수)
# ============================================================
import site
from pathlib import Path

def safe_read(p): return Path(p).read_text(encoding='utf-8')
def safe_write(p, c): Path(p).write_text(c, encoding='utf-8')

# 1) full_images_datamanager.py
for p in site.getsitepackages():
    cand = Path(p) / 'nerfstudio/data/datamanagers/full_images_datamanager.py'
    if cand.exists():
        code = safe_read(cand)
        old = '        data = self.cached_train[image_idx]'
        new = '''        if not hasattr(self, 'cached_train') or not self.cached_train:
            self.setup_train()
        data = self.cached_train[image_idx]'''
        if old in code:
            safe_write(cand, code.replace(old, new))
            print(f'full_images_datamanager 패치 완료')
        break

# 2) dn_datamanager.py
dm_path = Path('/content/dn-splatter/dn_splatter/dn_datamanager.py')
code = safe_read(dm_path)
old = '        data = deepcopy(self.cached_train[self.image_idx])'
new = '''        if hasattr(self, 'cached_train') and self.cached_train:
            data = deepcopy(self.cached_train[self.image_idx])
        else:
            data = super().next_train(step)
            return data'''
if old in code:
    safe_write(dm_path, code.replace(old, new))
    print('dn_datamanager 패치 완료')
else:
    print('dn_datamanager 이미 패치됨')


In [ ]:
# ============================================================
# Cell 3b: PLY export
# ============================================================
import subprocess, sys, os
from pathlib import Path

os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']

BASE       = '/content/drive/MyDrive/3dgs_project'
OUTPUT_DIR = f'{BASE}/output'

configs = sorted(Path(OUTPUT_DIR).rglob('config.yml'),
                 key=lambda x: x.stat().st_mtime)
assert configs, 'config.yml 없음'
CKPT_DIR = str(configs[-1].parent)
print(f'체크포인트: {CKPT_DIR}')

cmd = [sys.executable, '-m', 'nerfstudio.scripts.exporter',
       'gaussian-splat',
       '--load-config', f'{CKPT_DIR}/config.yml',
       '--output-dir', CKPT_DIR]

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout: print(line, end='')
proc.wait()
print(f'Return code: {proc.returncode}')

ply_files = list(Path(CKPT_DIR).rglob('*.ply'))
for f in ply_files:
    print(f'{f.stat().st_size/1e6:.1f} MB  {f}')

if ply_files:
    best = str(sorted(ply_files, key=lambda x: x.stat().st_size)[-1])
    Path(f'{OUTPUT_DIR}/logs/04_selected_ply_path.txt').write_text(best)
    print(f'OK: {best}')


In [ ]:
# ============================================================
# Cell 4: 학습 결과 .ply 확인
# ============================================================
import os, time
from pathlib import Path

BASE       = '/content/drive/MyDrive/3dgs_project'
OUTPUT_DIR = f'{BASE}/output'
LOG_DIR    = f'{OUTPUT_DIR}/logs'
os.makedirs(LOG_DIR, exist_ok=True)

ply_files = sorted(Path(OUTPUT_DIR).rglob('*.ply'),
                   key=lambda x: x.stat().st_mtime)
lines = [f'확인 시각: {time.strftime("%Y-%m-%d %H:%M:%S")}',
         f'발견된 .ply 수: {len(ply_files)}', '']
for f in ply_files:
    lines.append(f'{f}  ({f.stat().st_size/1e6:.1f} MB)')

if ply_files:
    GAUSSIAN_PLY = str(ply_files[-1])
    lines.append(f'\n-> 선택: {GAUSSIAN_PLY}')
    Path(f'{LOG_DIR}/04_selected_ply_path.txt').write_text(GAUSSIAN_PLY)
    print('OK:', GAUSSIAN_PLY)
else:
    print('ply 없음')

print('\n'.join(lines))
Path(f'{LOG_DIR}/04_ply_check.txt').write_text('\n'.join(lines))


In [ ]:
import subprocess, os, re, sys, time
from pathlib import Path
from IPython.display import clear_output

os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']

# ── 패치 ──────────────────────────────────────────────────────
def safe_read(p): return Path(p).read_text(encoding='utf-8')
def safe_write(p, c): Path(p).write_text(c, encoding='utf-8')

repo = '/content/dn-splatter'

# dn_model.py
dm = Path(f'{repo}/dn_splatter/dn_model.py')
lines = dm.read_text().split('\n')
if 'confidence_map=confidence,' in lines[719]:
    lines[719] = '                confidence_map=confidence if "confidence" in locals() else None,'
    dm.write_text('\n'.join(lines))
    print('dn_model.py 패치 완료')
else:
    print('dn_model.py 이미 패치됨')

# regularization_strategy.py
reg = Path(f'{repo}/dn_splatter/regularization_strategy.py')
lines = reg.read_text().split('\n')
for i, l in enumerate(lines):
    if l.strip() == 'gt_depth = torch.where(confidence_map > 0, gt_depth, 0).cuda()':
        indent = ' ' * (len(l) - len(l.lstrip()))
        lines[i:i+1] = [
            f'{indent}if confidence_map is not None:',
            f'{indent}    gt_depth = torch.where(confidence_map > 0, gt_depth, 0).cuda()',
            f'{indent}else:',
            f'{indent}    gt_depth = gt_depth.cuda()',
        ]
        reg.write_text('\n'.join(lines))
        print('regularization_strategy.py 패치 완료')
        break
else:
    print('regularization_strategy.py 이미 패치됨')

# eval_utils.py
import site
for p in site.getsitepackages():
    cand = f'{p}/nerfstudio/utils/eval_utils.py'
    if os.path.exists(cand):
        code = safe_read(cand)
        if 'weights_only=False' not in code:
            safe_write(cand, code.replace(
                'torch.load(load_path, map_location="cpu")',
                'torch.load(load_path, map_location="cpu", weights_only=False)'))
            print('eval_utils.py 패치 완료')
        else:
            print('eval_utils.py 이미 패치됨')
        break

# ── 실험 A ────────────────────────────────────────────────────
BASE       = '/content/drive/MyDrive/3dgs_project'
PROC_DIR   = f'{BASE}/processed'
OUTPUT_DIR = f'{BASE}/output'

EXP_NAME = f'exp_A_scale75_{time.strftime("%H%M%S")}'
MAX_ITER = 5000
LOG_EVERY = 100

cmd = [
    sys.executable, '-m', 'nerfstudio.scripts.train',
    'ags-mesh',
    '--output-dir', OUTPUT_DIR,
    '--experiment-name', EXP_NAME,
    '--max-num-iterations', str(MAX_ITER),
    '--vis', 'tensorboard',
    '--pipeline.model.use-depth-loss', 'True',
    '--pipeline.model.depth-lambda', '0.2',
    '--pipeline.model.use-normal-loss', 'True',
    '--pipeline.model.use-normal-tv-loss', 'True',
    '--pipeline.model.normal-supervision', 'depth',
    'normal-nerfstudio',
    '--data', PROC_DIR,
    '--load-3D-points', 'True',
    '--load-normals', 'False',
    '--load-depth-confidence-masks', 'False',
    '--downscale-factor', '1',
    '--scene-scale', '75.0',
    '--auto-scale-poses', 'False',
]

print(f'\n실험: {EXP_NAME}')
pat_step = re.compile(r'^\s*(\d+)\s*\(')
pat_loss = re.compile(r'loss[:\s]+([\d\.eE+\-]+)', re.I)
pat_psnr = re.compile(r'psnr[:\s]+([\d\.]+)', re.I)
last_step, last_loss, last_psnr = 0, '-', '-'
recent_lines = []
start_time = time.time()

def render_status():
    elapsed = time.time() - start_time
    pct = last_step / MAX_ITER * 100
    bar = '#' * int(pct/5) + '.' * (20 - int(pct/5))
    eta_str = time.strftime('%H:%M:%S', time.gmtime(
        elapsed / last_step * (MAX_ITER - last_step))) if last_step > 0 else '--:--:--'
    clear_output(wait=True)
    print(f'실험: {EXP_NAME}')
    print(f'  진행: [{bar}] {pct:5.1f}%  ({last_step:>5}/{MAX_ITER})')
    print(f'  경과: {time.strftime("%H:%M:%S", time.gmtime(elapsed))}   ETA: {eta_str}')
    print(f'  Loss: {last_loss:<12}  PSNR: {last_psnr}')
    print('최근 로그:')
    for l in recent_lines[-10:]: print(' ', l)

proc = subprocess.Popen(
    cmd, cwd='/content/dn-splatter',
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1)
for raw_line in proc.stdout:
    for line in raw_line.replace('\r', '\n').split('\n'):
        line = line.strip()
        if not line: continue
        recent_lines.append(line)
        if len(recent_lines) > 100: recent_lines.pop(0)
        m = pat_step.search(line)
        if m: last_step = int(m.group(1))
        m = pat_loss.search(line)
        if m: last_loss = m.group(1)
        m = pat_psnr.search(line)
        if m: last_psnr = m.group(1)
        if last_step > 0 and last_step % LOG_EVERY == 0:
            render_status()
proc.wait()
last_step = MAX_ITER; render_status()
print(f'\nReturn code: {proc.returncode}')

if proc.returncode == 0:
    import torch, numpy as np
    ckpts = [c for c in sorted(Path(OUTPUT_DIR).rglob('*.ckpt'),
             key=lambda x: x.stat().st_mtime) if EXP_NAME in str(c)]
    if ckpts:
        state = torch.load(str(ckpts[-1]), map_location='cpu', weights_only=False)
        means = state['pipeline']['_model.gauss_params.means'].numpy()
        print(f'\n[{EXP_NAME}] Gaussian 분포:')
        print(f'  N={len(means):,}')
        print(f'  X: [{means[:,0].min():.1f}, {means[:,0].max():.1f}]  range={np.ptp(means[:,0]):.1f}m')
        print(f'  Y: [{means[:,1].min():.1f}, {means[:,1].max():.1f}]  range={np.ptp(means[:,1]):.1f}m')
        print(f'  Z: [{means[:,2].min():.1f}, {means[:,2].max():.1f}]  range={np.ptp(means[:,2]):.1f}m')

실험: exp_A_scale75_050644
  진행: [####################] 100.0%  ( 5000/5000)
  경과: 00:15:54   ETA: 00:00:00
  Loss: -             PSNR: -
최근 로그:
  │   Config File          │ /content/drive/MyDrive/3dgs_project/output/exp_A_scale75_050644/ags-mesh/2026-05-18_0507…   │
  │   Checkpoint Directory │ /content/drive/MyDrive/3dgs_project/output/exp_A_scale75_050644/ags-mesh/2026-05-18_0507…   │
  │                        ╵                                                                                             │
  ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
  Printing profiling stats, from longest to shortest duration in seconds
  VanillaPipeline.get_eval_loss_dict: 6.8161
  Trainer.train_iteration: 0.1596
  VanillaPipeline.get_train_loss_dict: 0.1363
  VanillaPipeline.get_eval_image_metrics_and_images: 0.1122
  Trainer.eval_iteration: 0.0126

Return code: 0

[exp_A_scale75_050644] Gaussian 분포:
  N=115,831
  X: [-9.

In [ ]:
import json
from pathlib import Path

# dataparser_transforms 확인
BASE = '/content/drive/MyDrive/3dgs_project'
tf_files = sorted(
    Path(f'{BASE}/output/exp_A_scale75_050644').rglob('dataparser_transforms.json'),
    key=lambda x: x.stat().st_mtime
)
data = json.loads(tf_files[-1].read_text())
print(f'scale: {data["scale"]}')
import numpy as np
T = np.array(data['transform'])
print(f'transform:\n{T}')
print(f'\ntransform 의미:')
print(f'  새 X축 = 원래 {T[0,:3].round(2)}')
print(f'  새 Y축 = 원래 {T[1,:3].round(2)}')
print(f'  새 Z축 = 원래 {T[2,:3].round(2)}')

scale: 1.0
transform:
[[ 9.99793530e-01  1.46303577e-02  1.41004073e-02  2.18082237e+00]
 [ 1.46303577e-02 -3.67711782e-02 -9.99216616e-01  1.63324890e+01]
 [-1.41004073e-02  9.99216616e-01 -3.69776487e-02  3.03543210e-01]]

transform 의미:
  새 X축 = 원래 [1.   0.01 0.01]
  새 Y축 = 원래 [ 0.01 -0.04 -1.  ]
  새 Z축 = 원래 [-0.01  1.   -0.04]


In [ ]:
import subprocess, os, re, sys, time
from pathlib import Path
from IPython.display import clear_output

os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']

BASE       = '/content/drive/MyDrive/3dgs_project'
PROC_DIR   = f'{BASE}/processed'
OUTPUT_DIR = f'{BASE}/output'
os.makedirs(f'{OUTPUT_DIR}/logs', exist_ok=True)

EXP_NAME  = f'indoor_30k_{time.strftime("%H%M%S")}'
MAX_ITER  = 30000
LOG_EVERY = 50

cmd = [
    sys.executable, '-m', 'nerfstudio.scripts.train',
    'ags-mesh',
    '--output-dir', OUTPUT_DIR,
    '--experiment-name', EXP_NAME,
    '--max-num-iterations', str(MAX_ITER),
    '--vis', 'tensorboard',
    '--pipeline.model.use-depth-loss', 'True',
    '--pipeline.model.depth-lambda', '0.2',
    '--pipeline.model.use-normal-loss', 'True',
    '--pipeline.model.use-normal-tv-loss', 'True',
    '--pipeline.model.normal-supervision', 'depth',
    'normal-nerfstudio',
    '--data', PROC_DIR,
    '--load-3D-points', 'True',
    '--load-normals', 'False',
    '--load-depth-confidence-masks', 'False',
    '--downscale-factor', '1',
    '--scene-scale', '75.0',
    '--auto-scale-poses', 'False',
]

print(f'실험: {EXP_NAME}')
pat_step = re.compile(r'^\s*(\d+)\s*\(')
pat_loss = re.compile(r'loss[:\s]+([\d\.eE+\-]+)', re.I)
pat_psnr = re.compile(r'psnr[:\s]+([\d\.]+)', re.I)
last_step, last_loss, last_psnr = 0, '-', '-'
recent_lines = []
start_time = time.time()

def render_status():
    elapsed = time.time() - start_time
    pct = last_step / MAX_ITER * 100
    bar = '#' * int(pct/5) + '.' * (20 - int(pct/5))
    eta_str = time.strftime('%H:%M:%S', time.gmtime(
        elapsed / last_step * (MAX_ITER - last_step))) if last_step > 0 else '--:--:--'
    clear_output(wait=True)
    print(f'실험: {EXP_NAME}')
    print(f'  진행: [{bar}] {pct:5.1f}%  ({last_step:>6}/{MAX_ITER})')
    print(f'  경과: {time.strftime("%H:%M:%S", time.gmtime(elapsed))}   ETA: {eta_str}')
    print(f'  Loss: {last_loss:<12}  PSNR: {last_psnr}')
    print('최근 로그:')
    for l in recent_lines[-10:]: print(' ', l)

with open(f'{OUTPUT_DIR}/logs/30k_log.txt', 'w') as logf:
    proc = subprocess.Popen(
        cmd, cwd='/content/dn-splatter',
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1)
    for raw_line in proc.stdout:
        for line in raw_line.replace('\r', '\n').split('\n'):
            line = line.strip()
            if not line: continue
            logf.write(line + '\n'); logf.flush()
            recent_lines.append(line)
            if len(recent_lines) > 100: recent_lines.pop(0)
            m = pat_step.search(line)
            if m: last_step = int(m.group(1))
            m = pat_loss.search(line)
            if m: last_loss = m.group(1)
            m = pat_psnr.search(line)
            if m: last_psnr = m.group(1)
            if last_step > 0 and last_step % LOG_EVERY == 0:
                render_status()
    proc.wait()

last_step = MAX_ITER; render_status()
print(f'\n소요: {(time.time()-start_time)/60:.1f}분  Return code: {proc.returncode}')

if proc.returncode == 0:
    import torch, numpy as np
    ckpts = sorted([c for c in Path(OUTPUT_DIR).rglob('*.ckpt')
                    if EXP_NAME in str(c)], key=lambda x: x.stat().st_mtime)
    state = torch.load(str(ckpts[-1]), map_location='cpu', weights_only=False)
    means     = state['pipeline']['_model.gauss_params.means'].numpy()
    opacities = torch.sigmoid(
        state['pipeline']['_model.gauss_params.opacities']).numpy().flatten()
    print(f'\nGaussian 분포:')
    print(f'  N={len(means):,}')
    print(f'  X: [{means[:,0].min():.1f}, {means[:,0].max():.1f}]  range={np.ptp(means[:,0]):.1f}m')
    print(f'  Y: [{means[:,1].min():.1f}, {means[:,1].max():.1f}]  range={np.ptp(means[:,1]):.1f}m')
    print(f'  Z: [{means[:,2].min():.1f}, {means[:,2].max():.1f}]  range={np.ptp(means[:,2]):.1f}m')
    print(f'  opacity>0.5: {(opacities>0.5).sum():,} ({(opacities>0.5).mean()*100:.1f}%)')

실험: indoor_30k_122608
  진행: [####################] 100.0%  ( 30000/30000)
  경과: 03:22:09   ETA: 00:00:00
  Loss: -             PSNR: -
최근 로그:
  │   Config File          │ /content/drive/MyDrive/3dgs_project/output/indoor_30k_122608/ags-mesh/2026-05-18_122624/…   │
  │   Checkpoint Directory │ /content/drive/MyDrive/3dgs_project/output/indoor_30k_122608/ags-mesh/2026-05-18_122624/…   │
  │                        ╵                                                                                             │
  ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
  Printing profiling stats, from longest to shortest duration in seconds
  VanillaPipeline.get_eval_loss_dict: 0.4390
  Trainer.train_iteration: 0.3864
  VanillaPipeline.get_train_loss_dict: 0.3605
  VanillaPipeline.get_eval_image_metrics_and_images: 0.0733
  Trainer.eval_iteration: 0.0011

소요: 202.2분  Return code: 0

Gaussian 분포:
  N=253,999
  X: [-7.5, 5.4]  ran

In [ ]:
# ============================================================
# Final: 최종 Gaussian Splat 경로 확인
# ============================================================
from pathlib import Path
import shutil, os, time

BASE       = '/content/drive/MyDrive/3dgs_project'
OUTPUT_DIR = f'{BASE}/output'
LOG_DIR    = f'{OUTPUT_DIR}/logs'
os.makedirs(LOG_DIR, exist_ok=True)

ply_files = sorted(Path(OUTPUT_DIR).rglob('*.ply'), key=lambda x: x.stat().st_mtime)
assert ply_files, 'Gaussian Splat .ply 파일이 없습니다. 먼저 학습과 export 셀을 실행하세요.'

FINAL_SPLAT = ply_files[-1]
final_copy = Path(OUTPUT_DIR) / 'final_splat.ply'
shutil.copy2(FINAL_SPLAT, final_copy)

Path(f'{LOG_DIR}/final_splat_path.txt').write_text(str(final_copy))
print(f'최종 splat: {final_copy}')
print(f'크기: {final_copy.stat().st_size/1e6:.1f} MB')
